# 項目10: バックテスト

項目3〜7でシグナル・リスクモデルを逐次作成し、項目8・8.5・9の3つのアロケーション手法を同一条件で比較する。実データのみを対象にし、合成データは入れない。


## 必要データ形式

| 入力 | 形式 | 必須内容 |
|---|---|---|
| `stock_returns` | wide | index=営業日、columns=`security_id`、例 `1234.T` |
| `theme_basket_weights` | long | `effective_date`, `theme_id`, `security_id`, `weight` 必須。`available_date` 任意 |
| `barra_estimation_universe` | CSV long | `date`, `security_id`, `in_universe` 必須 |
| `data/temp/X_YYYYMMDD.pkl` | pickle | `{"date": Timestamp, "X": DataFrame(index=TSCODE, columns=factors)}` |
| `data/mapping.pkl` | wide pickle | index=`date`, columns=4桁コード、values=`TSCODE` |
| `cash_returns` | 任意 | index=営業日、value=日次キャッシュリターン。未指定なら0 |
| `theme_eligibility` | 任意 | index=`theme_id`、投資可能ならTrue |

`X_YYYYMMDD.pkl` と `mapping.pkl` は対象日と完全一致する行だけを使う。forward-fill は行わない。Barra回帰は同日エクスポージャによるOLSで、回帰ウェイトは全銘柄 `1.0` とする。


## バックテストの前提

シグナルはリバランス日 `t` の情報で作り、約定は `t + execution_lag_days` に行う。リバランス間は個別銘柄集約ウェイトを日次リターンで自然ドリフトさせる。turnover と取引費用はテーマウェイトではなく、集約後の個別銘柄ウェイトで計算する。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Dict, Literal, Mapping, Optional, Sequence, Tuple
import math

import numpy as np
import pandas as pd

try:
    from scipy.optimize import Bounds, LinearConstraint, linprog, minimize
except ImportError as exc:
    raise ImportError("scipy が必要です。プロジェクトのPython環境に scipy があるか確認してください。") from exc

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 240)


@dataclass
class BacktestConfig:
    # Backtest period and execution.
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    asof_date: str = "YYYY-MM-DD"
    rebalance_frequency: Literal["month_end", "weekly"] = "month_end"
    weekly_anchor: str = "W-FRI"
    execution_lag_days: int = 1
    allow_cash: bool = True
    cash_allowed: bool = True
    cash_return_default_daily: float = 0.0
    transaction_cost_bps_one_way: float = 0.0
    annualization: int = 252
    verbose: bool = True

    # Item 1/2 inputs and exposure construction.
    theme_exposure_mode: Literal["basket_weight", "membership", "rank_weight"] = "basket_weight"
    min_basket_coverage: float = 0.80
    pure_exposure_min_norm: float = 1e-10
    pinv_rcond: float = 1e-12
    orthogonality_tol: float = 1e-6
    x_dir: str = "data/temp"
    mapping_path: str = "data/mapping.pkl"

    # Item 3 coherence.
    lookback_days: int = 60
    min_constituents: int = 10
    min_pair_observations: int = 40
    coherence_z_threshold: float = 2.0
    bootstrap_iterations: int = 200
    block_size_days: int = 10
    random_seed: int = 42

    # Item 4 theme payoff.
    theme_return_ridge_alpha: float = 1e-6
    theme_exposure_lag_days: int = 1
    return_estimation_lookback_days: int = 252

    # Item 5 momentum.
    momentum_lookback_days: int = 60
    skip_recent_days: int = 1
    risk_power_N: float = 1.0
    score_scale_c: float = 1.0
    min_momentum_observations: int = 20
    risk_floor: float = 1e-12

    # Item 6 importance.
    use_coherence_filter: bool = True

    # Item 7 covariance.
    covariance_lookback_days: int = 252
    covariance_half_life_days: int = 252
    covariance_min_observations: int = 60
    cross_block_shrinkage: float = 1.0
    eigenvalue_floor: float = 1e-10
    covariance_ridge: float = 1e-10
    specific_variance_floor_daily: float = 1e-10

    # Item 8 unconstrained Lee-Liu.
    target_score_S_T: float = 0.10

    # Item 8.5 constrained allocation.
    theme_target_fraction: float = 0.50
    theme_cap: float = 0.25
    stock_cap: float = 0.05
    turnover_penalty_lambda: float = 0.0
    maxiter: int = 2000
    ftol: float = 1e-12

    # Item 9 direct alpha.
    kappa: float = 1.0
    risk_aversion_gamma: float = 10.0

    # Item 10 precomputed pkl inputs.
    precompute_dir: str = "data/phase_b/precomputed/item10"
    require_precomputed_inputs: bool = True
    force_recompute_precomputed: bool = False
    equivalence_atol: float = 1e-10
    precompute_version: str = "item10_precompute_v1"

    def validate(self) -> None:
        self.cash_allowed = self.allow_cash
        if self.rebalance_frequency not in {"month_end", "weekly"}:
            raise ValueError("rebalance_frequency must be month_end or weekly.")
        if self.execution_lag_days < 1:
            raise ValueError("execution_lag_days must be at least 1 to avoid same-day execution.")
        if self.theme_exposure_mode not in {"basket_weight", "membership", "rank_weight"}:
            raise ValueError("theme_exposure_mode must be basket_weight, membership, or rank_weight.")
        if not 0 <= self.min_basket_coverage <= 1:
            raise ValueError("min_basket_coverage must lie in [0, 1].")
        if self.pure_exposure_min_norm < 0:
            raise ValueError("pure_exposure_min_norm must be nonnegative.")
        if self.pinv_rcond <= 0:
            raise ValueError("pinv_rcond must be positive.")
        if self.lookback_days <= 1 or self.momentum_lookback_days <= 0 or self.covariance_lookback_days <= 1:
            raise ValueError("lookback settings must be positive.")
        if self.min_constituents < 2:
            raise ValueError("min_constituents must be at least 2.")
        if self.bootstrap_iterations <= 0:
            raise ValueError("bootstrap_iterations must be positive.")
        if self.theme_exposure_lag_days < 1:
            raise ValueError("theme_exposure_lag_days must be at least 1.")
        if self.theme_return_ridge_alpha < 0 or self.risk_power_N < 0:
            raise ValueError("ridge and risk power must be nonnegative.")
        if not 0 <= self.theme_target_fraction <= 1:
            raise ValueError("theme_target_fraction must lie in [0, 1].")
        if self.theme_cap <= 0 or self.stock_cap <= 0:
            raise ValueError("theme_cap and stock_cap must be positive.")
        if self.turnover_penalty_lambda < 0 or self.transaction_cost_bps_one_way < 0:
            raise ValueError("cost settings must be nonnegative.")
        if self.risk_aversion_gamma < 0:
            raise ValueError("risk_aversion_gamma must be nonnegative.")
        if self.equivalence_atol <= 0:
            raise ValueError("equivalence_atol must be positive.")


# Reused helper cells from prior item notebooks keep these annotation names.
ThemePurificationConfig = BacktestConfig
ThemeReturnConfig = BacktestConfig
CoherenceConfig = BacktestConfig
MomentumConfig = BacktestConfig
ImportanceConfig = BacktestConfig
CovarianceConfig = BacktestConfig
LeeLiuUnconstrainedConfig = BacktestConfig
ConstrainedAllocationConfig = BacktestConfig
DirectAlphaConfig = BacktestConfig

cfg = BacktestConfig(
    start_date=None,
    end_date=None,
    rebalance_frequency="month_end",
    execution_lag_days=1,
    allow_cash=True,
    cash_return_default_daily=0.0,
    transaction_cost_bps_one_way=0.0,
    theme_exposure_mode="basket_weight",
    min_basket_coverage=0.80,
    lookback_days=60,
    bootstrap_iterations=200,
    return_estimation_lookback_days=252,
    momentum_lookback_days=60,
    skip_recent_days=1,
    covariance_lookback_days=252,
    covariance_min_observations=60,
    target_score_S_T=0.10,
    theme_target_fraction=0.50,
    theme_cap=0.25,
    stock_cap=0.05,
    turnover_penalty_lambda=0.0,
    kappa=1.0,
    risk_aversion_gamma=10.0,
    precompute_dir="data/phase_b/precomputed/item10",
    require_precomputed_inputs=True,
    force_recompute_precomputed=False,
    equivalence_atol=1e-10,
    precompute_version="item10_precompute_v1",
    x_dir="data/temp",
    mapping_path="data/mapping.pkl",
)

paths = {
    "stock_returns": "data/stock_returns.csv",
    "theme_basket_weights": "data/theme_basket_weights.csv",
    "barra_estimation_universe": "data/barra_estimation_universe.csv",
    "cash_returns": None,
    "theme_eligibility": None,
}

existing_inputs = {
    # "stock_returns": stock_returns,
    # "theme_basket_weights": theme_basket_weights,
    # "barra_estimation_universe": barra_estimation_universe,
    # "mapping": mapping,
    # "cash_returns": cash_returns,
    # "theme_eligibility": theme_eligibility,
}

allocation_methods = ["lee_liu_unconstrained", "constrained_realworld", "direct_alpha"]


## データ読込・正規化


In [ ]:
def _read_table(path: str | Path, wide: bool = False) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt"}:
        df = pd.read_csv(path)
    elif suffix in {".pkl", ".pickle"}:
        df = pd.read_pickle(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    elif suffix == ".feather":
        df = pd.read_feather(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if wide:
        date_col = "date" if "date" in df.columns else df.columns[0]
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
    return df


def _to_naive_timestamp(value: Any) -> pd.Timestamp:
    ts = pd.Timestamp(value)
    if ts.tzinfo is not None:
        ts = ts.tz_localize(None)
    return ts.normalize()


def _format_code4(value: Any) -> str:
    if pd.isna(value):
        raise ValueError("4桁コードにNaNは使えません。")
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    if text.endswith(".T"):
        text = text[:-2]
    return text.zfill(4)


def _normalise_security_id(value: Any) -> str:
    code = _format_code4(value)
    return f"{code}.T"


def _normalise_tscode(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


def _normalise_wide(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index)).tz_localize(None).normalize()
    out = out[~out.index.duplicated(keep="last")].sort_index()
    out.columns = [_normalise_security_id(c) for c in out.columns]
    if pd.Index(out.columns).duplicated().any():
        dup = pd.Index(out.columns)[pd.Index(out.columns).duplicated()].unique().tolist()
        raise ValueError(f"{name} columns are duplicated after security_id normalization: {dup[:10]}")
    out = out.apply(pd.to_numeric, errors="coerce")
    if out.empty:
        raise ValueError(f"{name} is empty.")
    return out.astype(float)


def _normalise_long_dates(df: pd.DataFrame, date_cols: Sequence[str]) -> pd.DataFrame:
    out = df.copy()
    for col in date_cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col]).dt.tz_localize(None).dt.normalize()
    return out


def _coerce_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0.0) != 0.0
    return series.astype(str).str.strip().str.lower().isin({"true", "t", "1", "yes", "y"})


def load_mapping(path: str | Path) -> pd.DataFrame:
    mapping = pd.read_pickle(path)
    if not isinstance(mapping, pd.DataFrame):
        raise TypeError("data/mapping.pkl must be a pandas DataFrame.")
    out = mapping.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index)).tz_localize(None).normalize()
    out = out.sort_index()
    out.columns = [_format_code4(c) for c in out.columns]
    if pd.Index(out.columns).duplicated().any():
        dup = pd.Index(out.columns)[pd.Index(out.columns).duplicated()].unique().tolist()
        raise ValueError(f"mapping.pkl columns are duplicated after 4-digit normalization: {dup[:10]}")
    return out


def mapping_pairs_for_date(mapping: pd.DataFrame, asof_date: pd.Timestamp) -> pd.DataFrame:
    asof = _to_naive_timestamp(asof_date)
    matches = mapping.index[mapping.index == asof]
    if len(matches) == 0:
        raise KeyError(f"mapping.pkl に asof_date={asof.date()} の行がありません。")
    if len(matches) > 1:
        raise ValueError(f"mapping.pkl に asof_date={asof.date()} の重複行があります。")

    row = mapping.loc[matches[0]].dropna()
    pairs = pd.DataFrame({"code4": row.index.map(_format_code4), "TSCODE": row.map(_normalise_tscode).to_numpy()})
    pairs = pairs[pairs["TSCODE"] != ""].copy()
    pairs["security_id"] = pairs["code4"] + ".T"

    dup_tscode = pairs.loc[pairs["TSCODE"].duplicated(keep=False), "TSCODE"].unique().tolist()
    if dup_tscode:
        raise ValueError(f"mapping.pkl の当日行で同一TSCODEが複数の4桁コードに対応しています: {dup_tscode[:10]}")
    dup_sid = pairs.loc[pairs["security_id"].duplicated(keep=False), "security_id"].unique().tolist()
    if dup_sid:
        raise ValueError(f"mapping.pkl の当日行でsecurity_idが重複しています: {dup_sid[:10]}")
    return pairs


def load_daily_barra_exposure(asof_date: pd.Timestamp, x_dir: str | Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    asof = _to_naive_timestamp(asof_date)
    path = Path(x_dir) / f"X_{asof:%Y%m%d}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)

    payload = pd.read_pickle(path)
    if not isinstance(payload, dict) or "date" not in payload or "X" not in payload:
        raise ValueError("X_YYYYMMDD.pkl must contain {'date': Timestamp, 'X': DataFrame}.")
    payload_date = _to_naive_timestamp(payload["date"])
    if payload_date != asof:
        raise ValueError(f"pkl内date={payload_date.date()} が asof_date={asof.date()} と一致しません。")

    X = payload["X"]
    if not isinstance(X, pd.DataFrame):
        raise TypeError("payload['X'] must be a pandas DataFrame.")
    X = X.copy()
    X.index = X.index.map(_normalise_tscode)
    X.columns = X.columns.map(str)
    X = X.apply(pd.to_numeric, errors="coerce")
    if X.empty:
        raise ValueError(f"{path} のXが空です。")
    meta = {"x_path": path, "pkl_date": payload_date, "raw_tscode_count": X.shape[0], "factor_count": X.shape[1]}
    return X.astype(float), meta


def load_raw_inputs(paths: Mapping[str, str], cfg: ThemePurificationConfig) -> Dict[str, Any]:
    return {
        "stock_returns": _read_table(paths["stock_returns"], wide=True),
        "theme_basket_weights": _read_table(paths["theme_basket_weights"]),
        "barra_estimation_universe": _read_table(paths["barra_estimation_universe"]),
        "mapping": load_mapping(cfg.mapping_path),
    }


def validate_and_normalise_inputs(raw: Mapping[str, Any], cfg: ThemePurificationConfig) -> Dict[str, Any]:
    cfg.validate()

    stock_returns = _normalise_wide(raw["stock_returns"], "stock_returns")

    theme_basket_weights = _normalise_long_dates(raw["theme_basket_weights"], ["effective_date", "available_date"])
    required_bw = {"effective_date", "theme_id", "security_id", "weight"}
    missing = required_bw - set(theme_basket_weights.columns)
    if missing:
        raise ValueError(f"theme_basket_weights is missing columns: {sorted(missing)}")
    if "available_date" not in theme_basket_weights.columns:
        theme_basket_weights["available_date"] = theme_basket_weights["effective_date"]
    else:
        theme_basket_weights["available_date"] = theme_basket_weights["available_date"].fillna(
            theme_basket_weights["effective_date"]
        )
    theme_basket_weights["knowledge_date"] = theme_basket_weights[["effective_date", "available_date"]].max(axis=1)
    theme_basket_weights["theme_id"] = theme_basket_weights["theme_id"].astype(str)
    theme_basket_weights["security_id"] = theme_basket_weights["security_id"].map(_normalise_security_id)
    theme_basket_weights["weight"] = pd.to_numeric(theme_basket_weights["weight"], errors="coerce")
    if theme_basket_weights["weight"].isna().any():
        raise ValueError("theme_basket_weights.weight contains nonnumeric values.")
    if (theme_basket_weights["weight"] < -1e-12).any():
        raise ValueError("Theme basket weights must be nonnegative.")
    theme_basket_weights = theme_basket_weights[theme_basket_weights["weight"] > 0].copy()
    sums = theme_basket_weights.groupby(["knowledge_date", "theme_id"])["weight"].transform("sum")
    if (sums <= 0).any():
        raise ValueError("At least one theme snapshot has nonpositive total weight.")
    theme_basket_weights["weight"] = theme_basket_weights["weight"] / sums

    universe = _normalise_long_dates(raw["barra_estimation_universe"], ["date"])
    required_eu = {"date", "security_id", "in_universe"}
    missing = required_eu - set(universe.columns)
    if missing:
        raise ValueError(f"barra_estimation_universe is missing columns: {sorted(missing)}")
    universe["security_id"] = universe["security_id"].map(_normalise_security_id)
    universe["in_universe"] = _coerce_bool(universe["in_universe"])

    return {
        "stock_returns": stock_returns,
        "theme_basket_weights": theme_basket_weights,
        "barra_estimation_universe": universe,
        "mapping": raw["mapping"],
    }



def _normalise_date_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index)).tz_localize(None).normalize()
    out = out[~out.index.duplicated(keep="last")].sort_index()
    return out.apply(pd.to_numeric, errors="coerce")


def _existing_or_path(name: str, paths: Mapping[str, Optional[str]], existing_inputs: Mapping[str, Any], *, wide: bool = False) -> Any:
    if name in existing_inputs and existing_inputs[name] is not None:
        obj = existing_inputs[name]
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            return obj.copy()
        raise TypeError(f"existing_inputs['{name}'] must be a pandas DataFrame or Series.")
    path = paths.get(name)
    if not path:
        return None
    return _read_table(path, wide=wide)


def load_raw_inputs_from_paths_or_existing(
    paths: Mapping[str, Optional[str]],
    cfg: BacktestConfig,
    existing_inputs: Mapping[str, Any],
) -> Dict[str, Any]:
    stock_returns = _existing_or_path("stock_returns", paths, existing_inputs, wide=True)
    theme_basket_weights = _existing_or_path("theme_basket_weights", paths, existing_inputs, wide=False)
    barra_estimation_universe = _existing_or_path("barra_estimation_universe", paths, existing_inputs, wide=False)
    mapping = existing_inputs.get("mapping")
    if mapping is None:
        mapping = load_mapping(cfg.mapping_path)
    if stock_returns is None or theme_basket_weights is None or barra_estimation_universe is None:
        raise ValueError("stock_returns, theme_basket_weights, barra_estimation_universe は必須です。")
    return {
        "stock_returns": stock_returns,
        "theme_basket_weights": theme_basket_weights,
        "barra_estimation_universe": barra_estimation_universe,
        "mapping": mapping,
    }


def load_optional_cash_returns(
    paths: Mapping[str, Optional[str]],
    existing_inputs: Mapping[str, Any],
    stock_index: pd.DatetimeIndex,
    default_daily: float,
) -> pd.Series:
    obj = _existing_or_path("cash_returns", paths, existing_inputs, wide=True)
    if obj is None:
        return pd.Series(default_daily, index=stock_index, name="cash_return")
    if isinstance(obj, pd.Series):
        s = obj.copy()
    elif isinstance(obj, pd.DataFrame):
        if "cash_return" in obj.columns:
            s = obj["cash_return"].copy()
        elif obj.shape[1] == 1:
            s = obj.iloc[:, 0].copy()
        else:
            raise ValueError("cash_returns はSeries、または cash_return 列を持つDataFrameにしてください。")
    else:
        raise TypeError("cash_returns must be a pandas Series or DataFrame.")
    s.index = pd.DatetimeIndex(pd.to_datetime(s.index)).tz_localize(None).normalize()
    return pd.to_numeric(s, errors="coerce").reindex(stock_index).fillna(default_daily).rename("cash_return")


def load_optional_theme_eligibility(paths: Mapping[str, Optional[str]], existing_inputs: Mapping[str, Any]) -> Optional[pd.Series]:
    obj = _existing_or_path("theme_eligibility", paths, existing_inputs, wide=False)
    if obj is None:
        return None
    if isinstance(obj, pd.Series):
        return obj.copy()
    if isinstance(obj, pd.DataFrame):
        if "theme_id" in obj.columns:
            value_col = "eligible" if "eligible" in obj.columns else "is_eligible" if "is_eligible" in obj.columns else obj.columns[-1]
            return obj.set_index("theme_id")[value_col]
        if obj.shape[1] == 1:
            return obj.iloc[:, 0]
        if obj.shape[1] == 2:
            return obj.set_index(obj.columns[0]).iloc[:, 0]
    raise ValueError("theme_eligibility をSeriesとして解釈できません。")


## Point-in-time ストア


In [ ]:
class UniverseStore:
    def __init__(self, universe: pd.DataFrame):
        self.by_date: Dict[pd.Timestamp, pd.Series] = {}
        for d, g in universe.groupby("date", sort=True):
            if g["security_id"].duplicated().any():
                dup = g.loc[g["security_id"].duplicated(keep=False), "security_id"].unique().tolist()
                raise ValueError(f"Universe has duplicated security_id on {pd.Timestamp(d).date()}: {dup[:10]}")
            s = g.set_index("security_id")["in_universe"].astype(bool)
            self.by_date[pd.Timestamp(d)] = s
        self.dates = pd.DatetimeIndex(sorted(self.by_date))
        if len(self.dates) == 0:
            raise ValueError("UniverseStore requires at least one date.")

    def get_exact(self, asof: pd.Timestamp) -> pd.Series:
        asof = _to_naive_timestamp(asof)
        if asof not in self.by_date:
            raise KeyError(f"barra_estimation_universe に asof_date={asof.date()} の行がありません。")
        return self.by_date[asof]


class BasketStore:
    def __init__(self, basket_weights: pd.DataFrame):
        self.versions: Dict[str, list[Tuple[pd.Timestamp, pd.Series]]] = {}
        grouped = basket_weights.groupby(["theme_id", "knowledge_date"], sort=True)
        for (theme, d), g in grouped:
            s = g.groupby("security_id")["weight"].sum()
            s = s[s > 0]
            if s.sum() <= 0:
                continue
            s = s / s.sum()
            self.versions.setdefault(str(theme), []).append((pd.Timestamp(d), s.astype(float)))
        for theme in self.versions:
            self.versions[theme].sort(key=lambda x: x[0])
        self.themes = sorted(self.versions)

    def snapshot_with_dates(self, asof: pd.Timestamp) -> Tuple[pd.DataFrame, pd.Series]:
        asof = _to_naive_timestamp(asof)
        cols = {}
        dates = {}
        for theme, versions in self.versions.items():
            version_dates = pd.DatetimeIndex([v[0] for v in versions])
            pos = version_dates.searchsorted(asof, side="right") - 1
            if pos >= 0:
                cols[theme] = versions[pos][1]
                dates[theme] = versions[pos][0]
        if not cols:
            return pd.DataFrame(dtype=float), pd.Series(dtype="datetime64[ns]", name="basket_knowledge_date")
        a = pd.DataFrame(cols).fillna(0.0)
        sums = a.sum(axis=0)
        a = a.loc[:, sums > 0]
        a = a.div(a.sum(axis=0), axis=1).sort_index()
        date_series = pd.Series(dates, name="basket_knowledge_date").reindex(a.columns)
        return a, pd.to_datetime(date_series)


@dataclass
class Stores:
    estimation_universe: UniverseStore
    baskets: BasketStore


def build_stores(data: Mapping[str, Any]) -> Stores:
    return Stores(
        estimation_universe=UniverseStore(data["barra_estimation_universe"]),
        baskets=BasketStore(data["theme_basket_weights"]),
    )


## テーマ構成・Barra純化ロジック


In [ ]:
def map_barra_exposure_to_security_id(
    X_tscode: pd.DataFrame,
    mapping: pd.DataFrame,
    asof_date: pd.Timestamp,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    pairs = mapping_pairs_for_date(mapping, asof_date)
    tscode_to_security = pairs.set_index("TSCODE")["security_id"]

    tscode_index = pd.Index(X_tscode.index.map(_normalise_tscode), name="TSCODE")
    mapped_security = tscode_to_security.reindex(tscode_index)
    mapped_mask = mapped_security.notna().to_numpy()
    unmapped = pd.Index(tscode_index[~mapped_mask], name="unmapped_TSCODE")

    X_mapped = X_tscode.loc[mapped_mask].copy()
    X_mapped.index = pd.Index(mapped_security[mapped_mask].to_numpy(), name="security_id")
    if X_mapped.index.duplicated().any():
        dup = X_mapped.index[X_mapped.index.duplicated()].unique().tolist()
        raise ValueError(f"mapping後のBarraエクスポージャでsecurity_idが重複しています: {dup[:10]}")

    map_diag = {
        "mapping_date": _to_naive_timestamp(asof_date),
        "mapping_pair_count": len(pairs),
        "x_tscode_count": len(tscode_index),
        "mapping_success_count": int(mapped_mask.sum()),
        "mapping_failure_count": int((~mapped_mask).sum()),
        "mapped_security_id_is_unique": not X_mapped.index.duplicated().any(),
        "unmapped_tscodes": unmapped,
    }
    return X_mapped.sort_index(), pairs, map_diag


def build_raw_theme_exposure(
    basket_snapshot: pd.DataFrame,
    universe: Sequence[str],
    mode: str,
) -> pd.DataFrame:
    a = basket_snapshot.reindex(index=list(universe), fill_value=0.0).astype(float)
    if mode == "membership":
        x = (a > 0).astype(float)
    elif mode == "rank_weight":
        x = a.rank(axis=0, pct=True, method="average").where(a > 0, 0.0)
    elif mode == "basket_weight":
        x = a.copy()
    else:
        raise ValueError(f"Unknown theme_exposure_mode={mode}")

    for col in x.columns:
        v = x[col].to_numpy(dtype=float)
        valid = np.isfinite(v)
        if valid.sum() == 0:
            x[col] = 0.0
            continue
        norm = math.sqrt(max(float(np.sum(v[valid] ** 2)), 0.0))
        if norm > 0:
            x[col] = v / norm
    return x


def ols_column_norms(frame: pd.DataFrame) -> pd.Series:
    out = {}
    for col in frame.columns:
        v = frame[col].to_numpy(dtype=float)
        valid = np.isfinite(v)
        out[col] = math.sqrt(max(float(np.sum(v[valid] ** 2)), 0.0)) if valid.any() else np.nan
    return pd.Series(out, dtype=float, name="ols_norm")


def construct_theme_exposure_snapshot(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: ThemePurificationConfig,
) -> Dict[str, Any]:
    cfg.validate()
    asof = _to_naive_timestamp(cfg.asof_date)

    X_tscode, x_meta = load_daily_barra_exposure(asof, cfg.x_dir)
    X_full, mapping_pairs, mapping_diag = map_barra_exposure_to_security_id(X_tscode, data["mapping"], asof)

    universe = stores.estimation_universe.get_exact(asof)
    in_universe = universe[universe].index
    A_raw, basket_dates = stores.baskets.snapshot_with_dates(asof)
    if A_raw.empty:
        raise ValueError(f"No theme baskets available on or before {asof}.")

    return_universe = pd.Index(data["stock_returns"].columns)
    stock_x_overlap = return_universe.intersection(X_full.index)
    stock_x_universe = stock_x_overlap.intersection(in_universe)

    X_candidate = X_full.reindex(stock_x_universe)
    complete_x = X_candidate.notna().all(axis=1)
    eligible_secs = stock_x_universe[complete_x.to_numpy()]
    X = X_candidate.loc[eligible_secs]

    original_sum = A_raw.sum(axis=0).replace(0.0, np.nan)
    A_before_filter = A_raw.reindex(index=eligible_secs, fill_value=0.0)
    coverage_all = (A_before_filter.sum(axis=0) / original_sum).replace([np.inf, -np.inf], np.nan)

    keep_themes = coverage_all[coverage_all >= cfg.min_basket_coverage].index
    A_t = A_before_filter.loc[:, keep_themes]
    A_t = A_t.loc[:, A_t.sum(axis=0) > 0]
    if A_t.empty:
        raise ValueError("No theme passes min_basket_coverage after universe/exposure filters.")
    A_t = A_t.div(A_t.sum(axis=0), axis=1)
    coverage = coverage_all.reindex(A_t.columns)

    X_M_t = build_raw_theme_exposure(A_t, eligible_secs, cfg.theme_exposure_mode)
    regression_weights = pd.Series(1.0, index=eligible_secs, name="regression_weight")

    filter_counts = pd.Series(
        {
            "x_raw_tscode_rows": x_meta["raw_tscode_count"],
            "x_mapped_security_rows": len(X_full),
            "mapping_success_count": mapping_diag["mapping_success_count"],
            "mapping_failure_count": mapping_diag["mapping_failure_count"],
            "stock_return_columns": len(return_universe),
            "in_universe_true": len(in_universe),
            "stock_and_x_overlap": len(stock_x_overlap),
            "stock_x_universe_overlap": len(stock_x_universe),
            "dropped_for_missing_barra_exposure": int((~complete_x).sum()),
            "complete_barra_exposure": len(eligible_secs),
            "raw_basket_security_count": A_raw.shape[0],
            "eligible_basket_security_count": int((A_raw.index.isin(eligible_secs)).sum()),
        },
        name="n_securities",
    )

    theme_filter = pd.DataFrame(
        {
            "raw_weight_sum_before_filter": original_sum,
            "coverage_after_security_filters": coverage_all,
            "kept_after_coverage_filter": pd.Series(A_raw.columns.isin(A_t.columns), index=A_raw.columns),
            "basket_knowledge_date": basket_dates.reindex(A_raw.columns),
        }
    )
    theme_filter.index.name = "theme_id"

    date_diagnostics = pd.Series(
        {
            "asof_date": asof,
            "x_path": str(x_meta["x_path"]),
            "pkl_date": x_meta["pkl_date"],
            "mapping_path": str(Path(cfg.mapping_path)),
            "mapping_date": mapping_diag["mapping_date"],
            "latest_basket_knowledge_date_used": basket_dates.reindex(A_t.columns).max(),
            "raw_theme_count": A_raw.shape[1],
            "coverage_kept_theme_count": A_t.shape[1],
            "factor_count": x_meta["factor_count"],
        },
        name="value",
    )

    diagnostics = {
        "date": date_diagnostics,
        "security_filter_counts": filter_counts,
        "theme_filter": theme_filter,
        "A_column_sum": A_t.sum(axis=0).rename("A_column_sum"),
        "X_M_ols_norm": ols_column_norms(X_M_t),
        "mapping_pairs": mapping_pairs,
        "unmapped_tscodes": mapping_diag["unmapped_tscodes"],
        "barra_factor_missing_by_security": X_candidate.isna().sum(axis=1).rename("n_missing_factors"),
    }

    return {
        "asof_date": asof,
        "X": X,
        "A_raw": A_raw,
        "A_t": A_t,
        "coverage": coverage.rename("coverage"),
        "X_M_t": X_M_t,
        "regression_weights": regression_weights,
        "eligible_securities": pd.Index(eligible_secs),
        "diagnostics": diagnostics,
    }


In [ ]:
def _safe_condition_number(matrix: np.ndarray) -> float:
    try:
        return float(np.linalg.cond(matrix))
    except Exception:
        return float("nan")


def ols_r2_table(Y: pd.DataFrame, fitted: pd.DataFrame, n_parameters: int) -> pd.DataFrame:
    rows = []
    for col in Y.columns:
        y = Y[col].to_numpy(dtype=float)
        yhat = fitted[col].to_numpy(dtype=float)
        valid = np.isfinite(y) & np.isfinite(yhat)
        n = int(valid.sum())
        if n == 0:
            rows.append({"theme_id": col, "n_obs": n, "r2": np.nan, "adjusted_r2": np.nan})
            continue
        yv = y[valid]
        yhatv = yhat[valid]
        sse = float(np.sum((yv - yhatv) ** 2))
        sst = float(np.sum((yv - yv.mean()) ** 2))
        if sst <= 0:
            r2 = np.nan
            adj = np.nan
        else:
            r2 = 1.0 - sse / sst
            adj = 1.0 - (1.0 - r2) * (n - 1) / max(n - n_parameters - 1, 1)
        rows.append({"theme_id": col, "n_obs": n, "r2": r2, "adjusted_r2": adj})
    return pd.DataFrame(rows).set_index("theme_id")


def purify_theme_exposures(
    X_t: pd.DataFrame,
    X_M_t: pd.DataFrame,
    cfg: ThemePurificationConfig,
) -> Dict[str, Any]:
    common = X_t.index.intersection(X_M_t.index)
    X = X_t.loc[common].copy()
    Y = X_M_t.loc[common].copy()

    valid_rows = X.notna().all(axis=1) & Y.notna().all(axis=1)
    X = X.loc[valid_rows]
    Y = Y.loc[valid_rows]
    if X.empty or Y.empty:
        raise ValueError("No valid rows remain for purification.")

    Xmat = X.to_numpy(dtype=float)
    Ymat = Y.to_numpy(dtype=float)
    XtX = Xmat.T @ Xmat
    Theta = np.linalg.pinv(XtX, rcond=cfg.pinv_rcond) @ Xmat.T @ Ymat
    fitted_mat = Xmat @ Theta
    residual_mat = Ymat - fitted_mat

    fitted = pd.DataFrame(fitted_mat, index=X.index, columns=Y.columns)
    residual = pd.DataFrame(residual_mat, index=X.index, columns=Y.columns)
    pre_norm = ols_column_norms(residual).rename("pure_norm_before_rescale")
    valid_theme = pre_norm[(pre_norm > cfg.pure_exposure_min_norm) & np.isfinite(pre_norm)].index

    dropped = pd.DataFrame(index=Y.columns)
    dropped["pure_norm_before_rescale"] = pre_norm
    dropped["dropped"] = ~dropped.index.isin(valid_theme)
    dropped["reason"] = np.where(dropped["dropped"], "pure_norm_below_threshold", "kept")

    if len(valid_theme) == 0:
        raise ValueError("All themes were removed after Barra purification.")

    Z = residual.loc[:, valid_theme].copy()
    for col in Z.columns:
        Z[col] = Z[col] / pre_norm.loc[col]

    Theta_t = pd.DataFrame(Theta, index=X.columns, columns=Y.columns).loc[:, valid_theme]
    X_M_valid = Y.loc[:, valid_theme]
    fitted_valid = fitted.loc[:, valid_theme]
    purification_r2 = ols_r2_table(X_M_valid, fitted_valid, n_parameters=X.shape[1])
    pure_norm = pre_norm.reindex(valid_theme)

    orthogonality = X.T @ Z
    orthogonality.index.name = "factor_id"
    orthogonality.columns.name = "theme_id"
    orthogonality_max_abs = float(np.nanmax(np.abs(orthogonality.to_numpy(dtype=float)))) if not orthogonality.empty else np.nan

    model_diag = pd.Series(
        {
            "n_securities": X.shape[0],
            "n_factors": X.shape[1],
            "n_themes_before_purification": Y.shape[1],
            "n_themes_after_purification": Z.shape[1],
            "x_rank": int(np.linalg.matrix_rank(Xmat)),
            "x_condition_number": _safe_condition_number(Xmat),
            "xtx_condition_number": _safe_condition_number(XtX),
            "orthogonality_max_abs": orthogonality_max_abs,
            "pinv_rcond": cfg.pinv_rcond,
            "pure_exposure_min_norm": cfg.pure_exposure_min_norm,
        },
        name="value",
    )

    corr_before = X_M_valid.corr()
    corr_after = Z.corr()

    return {
        "X_t": X,
        "X_M_t": X_M_valid,
        "Theta_t": Theta_t,
        "fitted_X_M_t": fitted_valid,
        "Z_t": Z,
        "purification_r2": purification_r2,
        "pure_norm": pure_norm,
        "dropped_themes": dropped,
        "orthogonality": orthogonality,
        "model_diagnostics": model_diag,
        "corr_before": corr_before,
        "corr_after": corr_after,
    }


## 日次Barra OLSと純化テーマ・リターン


In [ ]:
def previous_trading_date(index: pd.DatetimeIndex, date: pd.Timestamp, lag: int = 1) -> pd.Timestamp:
    dates = pd.DatetimeIndex(index).sort_values()
    date = _to_naive_timestamp(date)
    pos = dates.searchsorted(date, side="left")
    target = pos - lag
    if target < 0:
        raise KeyError(f"{date.date()} の {lag} 営業日前が stock_returns.index に存在しません。")
    return pd.Timestamp(dates[target])


def run_daily_barra_ols(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: Any,
    date: pd.Timestamp,
) -> Dict[str, Any]:
    d = _to_naive_timestamp(date)
    X_tscode, x_meta = load_daily_barra_exposure(d, cfg.x_dir)
    X_full, mapping_pairs, mapping_diag = map_barra_exposure_to_security_id(X_tscode, data["mapping"], d)
    universe = stores.estimation_universe.get_exact(d)
    in_universe = universe[universe].index

    if d not in data["stock_returns"].index:
        raise KeyError(f"stock_returns に date={d.date()} がありません。")
    y_all = data["stock_returns"].loc[d]

    candidates = pd.Index(y_all.index).intersection(X_full.index).intersection(in_universe)
    X_candidate = X_full.reindex(candidates)
    y_candidate = y_all.reindex(candidates)
    valid = y_candidate.notna() & X_candidate.notna().all(axis=1)
    X = X_candidate.loc[valid].astype(float)
    y = y_candidate.loc[valid].astype(float)
    if X.empty or len(y) == 0:
        raise ValueError(f"{d.date()} のBarra OLSに使える銘柄がありません。")

    Xmat = X.to_numpy(dtype=float)
    yvec = y.to_numpy(dtype=float)
    beta = np.linalg.pinv(Xmat, rcond=cfg.pinv_rcond) @ yvec
    fitted = Xmat @ beta
    residual = yvec - fitted

    factor_return = pd.Series(beta, index=X.columns, name=d)
    residual_s = pd.Series(residual, index=X.index, name=d)
    fitted_s = pd.Series(fitted, index=X.index, name=d)
    diag = pd.Series(
        {
            "date": d,
            "status": "ok",
            "x_path": str(x_meta["x_path"]),
            "pkl_date": x_meta["pkl_date"],
            "mapping_date": mapping_diag["mapping_date"],
            "x_raw_tscode_rows": x_meta["raw_tscode_count"],
            "mapping_success_count": mapping_diag["mapping_success_count"],
            "mapping_failure_count": mapping_diag["mapping_failure_count"],
            "stock_return_columns": len(y_all),
            "in_universe_true": len(in_universe),
            "ols_n_securities": len(y),
            "ols_n_factors": X.shape[1],
            "ols_rank": int(np.linalg.matrix_rank(Xmat)),
            "dropped_for_missing_return_or_exposure": int((~valid).sum()),
        },
        name=d,
    )
    return {
        "date": d,
        "X_t": X,
        "y_t": y,
        "factor_returns_f": factor_return,
        "fitted_returns": fitted_s,
        "residuals_u": residual_s,
        "diagnostics": diag,
        "mapping_pairs": mapping_pairs,
    }


def build_daily_barra_ols_panel(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: Any,
    dates: Sequence[pd.Timestamp],
) -> Dict[str, Any]:
    residual_rows = []
    fitted_rows = []
    factor_rows = []
    diag_rows = []
    errors = []
    for d in pd.DatetimeIndex(dates):
        try:
            result = run_daily_barra_ols(data, stores, cfg, d)
        except Exception as exc:
            errors.append({"date": _to_naive_timestamp(d), "status": f"error:{type(exc).__name__}", "message": str(exc)})
            continue
        residual_rows.append(result["residuals_u"])
        fitted_rows.append(result["fitted_returns"])
        factor_rows.append(result["factor_returns_f"])
        diag_rows.append(result["diagnostics"])
    if not residual_rows:
        raise ValueError("Barra OLSの成功日がありません。diagnostics_errors を確認してください。")
    daily_barra_residuals = pd.DataFrame(residual_rows).sort_index()
    daily_barra_fitted = pd.DataFrame(fitted_rows).sort_index()
    barra_factor_returns_f = pd.DataFrame(factor_rows).sort_index()
    daily_ols_diagnostics = pd.DataFrame(diag_rows).sort_index()
    diagnostics_errors = pd.DataFrame(errors).set_index("date") if errors else pd.DataFrame(columns=["status", "message"])
    return {
        "daily_barra_residuals": daily_barra_residuals,
        "daily_barra_fitted": daily_barra_fitted,
        "barra_factor_returns_f": barra_factor_returns_f,
        "daily_ols_diagnostics": daily_ols_diagnostics,
        "diagnostics_errors": diagnostics_errors,
    }


In [ ]:
def compute_purified_snapshot_for_date(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: ThemeReturnConfig,
    asof_date: pd.Timestamp,
) -> Dict[str, Any]:
    cfg_for_date = replace(cfg, asof_date=f"{_to_naive_timestamp(asof_date):%Y-%m-%d}")
    base_snapshot = construct_theme_exposure_snapshot(data, stores, cfg_for_date)
    purified = purify_theme_exposures(base_snapshot["X"], base_snapshot["X_M_t"], cfg_for_date)
    Z_t = purified["Z_t"]
    A_t = base_snapshot["A_t"].reindex(index=Z_t.index, columns=Z_t.columns).fillna(0.0)
    return {
        "asof_date": base_snapshot["asof_date"],
        "A_t": A_t,
        "X_t": purified["X_t"],
        "X_M_t": purified["X_M_t"],
        "Z_t": Z_t,
        "purification_r2": purified["purification_r2"],
        "pure_norm": purified["pure_norm"],
        "diagnostics": {**base_snapshot["diagnostics"], "purification_model": purified["model_diagnostics"]},
    }


def estimate_theme_return_for_date(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: ThemeReturnConfig,
    return_date: pd.Timestamp,
) -> Dict[str, Any]:
    d = _to_naive_timestamp(return_date)
    barra = run_daily_barra_ols(data, stores, cfg, d)
    snapshot_date = previous_trading_date(data["stock_returns"].index, d, cfg.theme_exposure_lag_days)
    snap = compute_purified_snapshot_for_date(data, stores, cfg, snapshot_date)

    u = barra["residuals_u"]
    Z = snap["Z_t"]
    common = Z.index.intersection(u.index)
    Z = Z.loc[common]
    u = u.reindex(common)
    valid = u.notna() & Z.notna().all(axis=1)
    Z = Z.loc[valid]
    u = u.loc[valid]
    if Z.empty or len(u) == 0:
        raise ValueError(f"{d.date()} のテーマリターン推定に使える銘柄がありません。")

    Zmat = Z.to_numpy(dtype=float)
    uvec = u.to_numpy(dtype=float)
    K = Z.shape[1]
    lhs = Zmat.T @ Zmat + cfg.theme_return_ridge_alpha * np.eye(K)
    rhs = Zmat.T @ uvec
    g = np.linalg.pinv(lhs, rcond=cfg.pinv_rcond) @ rhs
    fitted_theme = Zmat @ g
    e = uvec - fitted_theme

    denom = np.sum(Zmat * Zmat, axis=0)
    g_uni = np.divide(rhs, denom, out=np.full(K, np.nan), where=denom > 0)
    z_norm = pd.Series(np.sqrt(denom), index=Z.columns, name=d)

    diag = pd.Series(
        {
            "return_date": d,
            "snapshot_date_for_Z": snapshot_date,
            "n_securities": len(u),
            "n_themes": K,
            "z_rank": int(np.linalg.matrix_rank(Zmat)),
            "ridge_alpha": cfg.theme_return_ridge_alpha,
            "max_abs_z_norm_error": float(np.nanmax(np.abs(z_norm - 1.0))) if len(z_norm) else np.nan,
            "barra_ols_n_securities": barra["diagnostics"].loc["ols_n_securities"],
            "barra_ols_rank": barra["diagnostics"].loc["ols_rank"],
        },
        name=d,
    )
    return {
        "date": d,
        "snapshot_date": snapshot_date,
        "theme_returns_g": pd.Series(g, index=Z.columns, name=d),
        "univariate_theme_returns_g": pd.Series(g_uni, index=Z.columns, name=d),
        "theme_residuals_e": pd.Series(e, index=Z.index, name=d),
        "barra_residuals_u": u.rename(d),
        "barra_factor_returns_f": barra["factor_returns_f"],
        "z_norm": z_norm,
        "diagnostics": diag,
    }


def estimate_theme_return_history(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: ThemeReturnConfig,
) -> Dict[str, Any]:
    cfg.validate()
    asof = _to_naive_timestamp(cfg.asof_date)
    dates = data["stock_returns"].index[data["stock_returns"].index <= asof]
    dates = dates[-cfg.return_estimation_lookback_days :]
    rows_g = []
    rows_g_uni = []
    rows_e = []
    rows_u = []
    rows_f = []
    rows_diag = []
    rows_norm = []
    errors = []
    for d in dates:
        try:
            result = estimate_theme_return_for_date(data, stores, cfg, d)
        except Exception as exc:
            errors.append({"date": _to_naive_timestamp(d), "status": f"error:{type(exc).__name__}", "message": str(exc)})
            continue
        rows_g.append(result["theme_returns_g"])
        rows_g_uni.append(result["univariate_theme_returns_g"])
        rows_e.append(result["theme_residuals_e"])
        rows_u.append(result["barra_residuals_u"])
        rows_f.append(result["barra_factor_returns_f"])
        rows_diag.append(result["diagnostics"])
        rows_norm.append(result["z_norm"])
    if not rows_g:
        raise ValueError("純化テーマ・リターン推定の成功日がありません。diagnostics_errors を確認してください。")
    return {
        "theme_returns_g": pd.DataFrame(rows_g).sort_index(),
        "univariate_theme_returns_g": pd.DataFrame(rows_g_uni).sort_index(),
        "theme_residuals_e": pd.DataFrame(rows_e).sort_index(),
        "barra_residuals_u": pd.DataFrame(rows_u).sort_index(),
        "barra_factor_returns_f": pd.DataFrame(rows_f).sort_index(),
        "z_norms": pd.DataFrame(rows_norm).sort_index(),
        "theme_return_diagnostics": pd.DataFrame(rows_diag).sort_index(),
        "diagnostics_errors": pd.DataFrame(errors).set_index("date") if errors else pd.DataFrame(columns=["status", "message"]),
    }


## コヒーレンス・モメンタム・重要度


In [ ]:
def _pairwise_corr_values(frame: pd.DataFrame, min_obs: int) -> np.ndarray:
    x = frame.apply(pd.to_numeric, errors="coerce")
    keep = x.notna().sum(axis=0) >= min_obs
    x = x.loc[:, keep]
    if x.shape[1] < 2:
        return np.array([], dtype=float)
    corr = x.corr(min_periods=min_obs).to_numpy(dtype=float)
    iu = np.triu_indices_from(corr, k=1)
    vals = corr[iu]
    return vals[np.isfinite(vals)]


def mean_pairwise_corr(frame: pd.DataFrame, min_obs: int) -> float:
    vals = _pairwise_corr_values(frame, min_obs=min_obs)
    return float(vals.mean()) if len(vals) else np.nan


def benjamini_hochberg(pvals: pd.Series) -> pd.Series:
    p = pvals.dropna().clip(0, 1)
    if p.empty:
        return pd.Series(np.nan, index=pvals.index, dtype=float)
    order = np.argsort(p.to_numpy())
    ranked = p.to_numpy()[order]
    m = len(ranked)
    q = ranked * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out = pd.Series(np.nan, index=pvals.index, dtype=float)
    out.loc[p.index[order]] = q
    return out


def theme_member_table(A_t: pd.DataFrame, residuals: pd.DataFrame, cfg: CoherenceConfig) -> pd.DataFrame:
    rows = []
    available = pd.Index(residuals.columns)
    for theme in A_t.columns:
        members = A_t.index[(A_t[theme] > 0) & A_t.index.isin(available)]
        vals = _pairwise_corr_values(residuals.reindex(columns=members), cfg.min_pair_observations)
        rows.append(
            {
                "theme_id": theme,
                "n_constituents": len(members),
                "n_valid_pairs": len(vals),
                "passes_min_constituents": len(members) >= cfg.min_constituents,
                "passes_min_pair_observations": len(vals) > 0,
            }
        )
    return pd.DataFrame(rows).set_index("theme_id")


def _bootstrap_blocks(n_obs: int, block_size: int) -> list[np.ndarray]:
    return [np.arange(i, min(i + block_size, n_obs)) for i in range(0, n_obs, block_size)]


def _permute_residual_blocks_by_security(residuals: pd.DataFrame, cfg: CoherenceConfig, rng: np.random.Generator) -> pd.DataFrame:
    values = residuals.to_numpy(dtype=float)
    out = np.empty_like(values)
    blocks = _bootstrap_blocks(values.shape[0], cfg.block_size_days)
    for j in range(values.shape[1]):
        order = rng.permutation(len(blocks))
        out[:, j] = np.concatenate([values[blocks[k], j] for k in order])[: values.shape[0]]
    return pd.DataFrame(out, index=residuals.index, columns=residuals.columns)


def run_coherence_detection(
    A_t: pd.DataFrame,
    daily_barra_residuals: pd.DataFrame,
    cfg: CoherenceConfig,
) -> Dict[str, Any]:
    member_diag = theme_member_table(A_t, daily_barra_residuals, cfg)
    eligible_themes = member_diag.index[
        member_diag["passes_min_constituents"] & member_diag["passes_min_pair_observations"]
    ]
    observed = pd.Series(index=eligible_themes, dtype=float, name="apc")
    for theme in eligible_themes:
        members = A_t.index[(A_t[theme] > 0) & A_t.index.isin(daily_barra_residuals.columns)]
        observed.loc[theme] = mean_pairwise_corr(daily_barra_residuals.reindex(columns=members), cfg.min_pair_observations)

    rng = np.random.default_rng(cfg.random_seed + int(pd.Timestamp(cfg.asof_date).strftime("%Y%m%d")))
    null = pd.DataFrame(index=range(cfg.bootstrap_iterations), columns=eligible_themes, dtype=float)
    for b in range(cfg.bootstrap_iterations):
        boot = _permute_residual_blocks_by_security(daily_barra_residuals, cfg, rng)
        for theme in eligible_themes:
            members = A_t.index[(A_t[theme] > 0) & A_t.index.isin(boot.columns)]
            null.loc[b, theme] = mean_pairwise_corr(boot.reindex(columns=members), cfg.min_pair_observations)

    mu = null.mean(axis=0)
    sd = null.std(axis=0, ddof=1).replace(0.0, np.nan)
    z = (observed - mu) / sd
    p = pd.Series(index=eligible_themes, dtype=float, name="p")
    for theme in eligible_themes:
        vals = null[theme].dropna()
        p.loc[theme] = (1.0 + float((vals >= observed.loc[theme]).sum())) / (1.0 + len(vals)) if len(vals) else np.nan
    q = benjamini_hochberg(p)

    coherence = pd.DataFrame(index=A_t.columns)
    coherence["apc"] = observed.reindex(A_t.columns)
    coherence["bootstrap_mean"] = mu.reindex(A_t.columns)
    coherence["bootstrap_sd"] = sd.reindex(A_t.columns)
    coherence["z"] = z.reindex(A_t.columns)
    coherence["p"] = p.reindex(A_t.columns)
    coherence["q"] = q.reindex(A_t.columns)
    coherence = coherence.join(member_diag)
    coherence["coherent"] = coherence["z"] >= cfg.coherence_z_threshold
    coherence["exclusion_reason"] = "kept"
    coherence.loc[~coherence["passes_min_constituents"], "exclusion_reason"] = "min_constituents_not_met"
    coherence.loc[
        coherence["passes_min_constituents"] & ~coherence["passes_min_pair_observations"], "exclusion_reason"
    ] = "min_pair_observations_not_met"
    coherence.loc[coherence["z"].isna() & (coherence["exclusion_reason"] == "kept"), "exclusion_reason"] = "bootstrap_stat_unavailable"
    return {"coherence": coherence, "bootstrap_null": null, "member_diagnostics": member_diag}


In [ ]:
def compute_residual_momentum_scores(theme_returns_g: pd.DataFrame, cfg: MomentumConfig) -> Dict[str, Any]:
    cfg.validate()
    g = _normalise_date_index(theme_returns_g)
    asof = _to_naive_timestamp(cfg.asof_date)
    hist = g.loc[g.index <= asof]
    end = len(hist) - cfg.skip_recent_days
    start = end - cfg.momentum_lookback_days
    if start < 0 or end <= start:
        raise ValueError("momentum_lookback_days + skip_recent_days を満たす履歴が不足しています。")
    window = hist.iloc[start:end]
    obs = window.notna().sum(axis=0)
    raw_momentum = (1.0 + window).prod(axis=0, min_count=cfg.min_momentum_observations) - 1.0
    risk = window.std(axis=0, ddof=1) * math.sqrt(len(window))
    risk = risk.where(risk > cfg.risk_floor)
    score = cfg.score_scale_c * raw_momentum / (risk ** cfg.risk_power_N)
    score = score.replace([np.inf, -np.inf], np.nan)

    components = pd.DataFrame(
        {
            "raw_momentum": raw_momentum,
            "risk": risk,
            "observations": obs,
            "score": score,
            "kept": score.notna() & (obs >= cfg.min_momentum_observations),
        }
    )
    components["exclusion_reason"] = "kept"
    components.loc[obs < cfg.min_momentum_observations, "exclusion_reason"] = "insufficient_observations"
    components.loc[risk.isna() & (components["exclusion_reason"] == "kept"), "exclusion_reason"] = "zero_or_missing_risk"
    components.loc[score.isna() & (components["exclusion_reason"] == "kept"), "exclusion_reason"] = "missing_score"
    momentum_scores_q = score.where(components["kept"]).rename("momentum_score_q")
    diagnostics = pd.Series(
        {
            "asof_date": asof,
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            "window_rows": len(window),
            "skip_recent_days": cfg.skip_recent_days,
            "theme_count": g.shape[1],
            "kept_theme_count": int(components["kept"].sum()),
        },
        name="value",
    )
    return {
        "momentum_scores_q": momentum_scores_q,
        "momentum_components": components,
        "momentum_window": window,
        "diagnostics": diagnostics,
    }


def compute_long_only_theme_importance(
    momentum_scores_q: pd.Series,
    coherence: Optional[pd.DataFrame],
    cfg: ImportanceConfig,
) -> Dict[str, Any]:
    q = pd.to_numeric(momentum_scores_q.copy(), errors="coerce").rename("momentum_score_q")
    diagnostics = pd.DataFrame(index=q.index)
    diagnostics["score"] = q
    diagnostics["coherence_allowed"] = True
    if cfg.use_coherence_filter:
        if coherence is None:
            raise ValueError("use_coherence_filter=True の場合は coherence を指定してください。")
        coh = coherence.copy()
        if "coherent" not in coh.columns:
            raise ValueError("coherence must include a 'coherent' column.")
        diagnostics["coherence_allowed"] = coh["coherent"].reindex(q.index).fillna(False).astype(bool)
    q_filtered = q.where(diagnostics["coherence_allowed"], np.nan)
    positive_scores_p = q_filtered.clip(lower=0.0).fillna(0.0).rename("positive_score_p")
    if positive_scores_p.sum() > 0:
        theme_importance_s = (positive_scores_p / positive_scores_p.sum()).rename("theme_importance_s")
    else:
        theme_importance_s = pd.Series(0.0, index=q.index, name="theme_importance_s")
    diagnostics["positive_score_p"] = positive_scores_p
    diagnostics["importance_s"] = theme_importance_s
    diagnostics["exclusion_reason"] = "kept"
    diagnostics.loc[~diagnostics["coherence_allowed"], "exclusion_reason"] = "not_coherent"
    diagnostics.loc[diagnostics["coherence_allowed"] & q.isna(), "exclusion_reason"] = "missing_score"
    diagnostics.loc[diagnostics["coherence_allowed"] & q.notna() & (q <= 0), "exclusion_reason"] = "nonpositive_score"
    summary = pd.Series(
        {
            "use_coherence_filter": cfg.use_coherence_filter,
            "theme_count": len(q),
            "positive_theme_count": int((positive_scores_p > 0).sum()),
            "positive_score_sum": float(positive_scores_p.sum()),
            "importance_sum": float(theme_importance_s.sum()),
        },
        name="value",
    )
    return {
        "theme_importance_s": theme_importance_s,
        "positive_scores_p": positive_scores_p,
        "importance_diagnostics": diagnostics,
        "summary": summary,
    }


## 拡張共分散行列


In [ ]:
def _ewma_weights(n: int, half_life: float) -> np.ndarray:
    lam = math.exp(math.log(0.5) / half_life)
    w = lam ** np.arange(n - 1, -1, -1)
    return w / w.sum()


def ewma_cov_pairwise(frame: pd.DataFrame, half_life: int, min_obs: int) -> pd.DataFrame:
    x = _normalise_date_index(frame)
    cols = list(x.columns)
    out = pd.DataFrame(np.nan, index=cols, columns=cols, dtype=float)
    for i, ci in enumerate(cols):
        for j, cj in enumerate(cols[i:], start=i):
            pair = x[[ci, cj]].dropna()
            if len(pair) < min_obs:
                continue
            w = _ewma_weights(len(pair), half_life)
            arr = pair.to_numpy(dtype=float)
            mean = np.sum(arr * w[:, None], axis=0)
            centered = arr - mean
            cov = float(np.sum(w * centered[:, 0] * centered[:, 1]))
            out.loc[ci, cj] = cov
            out.loc[cj, ci] = cov
    return out


def ewma_variance(frame: pd.DataFrame, half_life: int, min_obs: int, floor: float) -> pd.Series:
    x = _normalise_date_index(frame)
    out = pd.Series(index=x.columns, dtype=float)
    for col in x.columns:
        s = x[col].dropna()
        if len(s) < min_obs:
            continue
        w = _ewma_weights(len(s), half_life)
        vals = s.to_numpy(dtype=float)
        mean = float(np.sum(w * vals))
        out.loc[col] = max(float(np.sum(w * (vals - mean) ** 2)), floor)
    return out


def nearest_psd(matrix: np.ndarray, floor: float) -> Tuple[np.ndarray, Dict[str, float]]:
    sym = 0.5 * (matrix + matrix.T)
    eigval, eigvec = np.linalg.eigh(sym)
    min_before = float(eigval.min()) if len(eigval) else np.nan
    clipped = np.maximum(eigval, floor)
    adjusted = (eigvec * clipped) @ eigvec.T
    adjusted = 0.5 * (adjusted + adjusted.T)
    return adjusted, {"min_eigenvalue_before_floor": min_before, "min_eigenvalue_after_floor": float(clipped.min())}


def build_augmented_basket_covariance(
    A_t: pd.DataFrame,
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    barra_factor_returns_f: pd.DataFrame,
    theme_returns_g: pd.DataFrame,
    theme_residuals_e: pd.DataFrame,
    cfg: CovarianceConfig,
) -> Dict[str, Any]:
    cfg.validate()
    asof = _to_naive_timestamp(cfg.asof_date)
    A = A_t.copy().astype(float)
    X = X_t.copy().astype(float)
    Z = Z_t.copy().astype(float)
    factor_returns = _normalise_date_index(barra_factor_returns_f)
    theme_returns = _normalise_date_index(theme_returns_g)
    residuals = _normalise_date_index(theme_residuals_e)

    securities = A.index.intersection(X.index).intersection(Z.index)
    themes = A.columns.intersection(Z.columns).intersection(theme_returns.columns)
    factor_cols = X.columns.intersection(factor_returns.columns)
    if len(securities) == 0 or len(themes) == 0 or len(factor_cols) == 0:
        raise ValueError("A_t, X_t, Z_t, returns の共通集合が空です。")
    A = A.loc[securities, themes]
    X = X.loc[securities, factor_cols]
    Z = Z.loc[securities, themes]

    f_hist = factor_returns.loc[factor_returns.index <= asof, factor_cols].tail(cfg.covariance_lookback_days)
    g_hist = theme_returns.loc[theme_returns.index <= asof, themes].tail(cfg.covariance_lookback_days)
    combined = pd.concat([f_hist, g_hist], axis=1).sort_index()
    F_raw = ewma_cov_pairwise(combined, cfg.covariance_half_life_days, cfg.covariance_min_observations)
    F_raw = F_raw.fillna(0.0)
    F = F_raw.to_numpy(dtype=float)
    p = len(factor_cols)
    F[:p, p:] *= cfg.cross_block_shrinkage
    F[p:, :p] *= cfg.cross_block_shrinkage
    F, psd_diag_f = nearest_psd(F, cfg.eigenvalue_floor)
    F += cfg.covariance_ridge * np.eye(F.shape[0])
    F_aug = pd.DataFrame(F, index=list(factor_cols) + list(themes), columns=list(factor_cols) + list(themes))

    e_hist = residuals.loc[residuals.index <= asof].reindex(columns=securities).tail(cfg.covariance_lookback_days)
    specific_variance = ewma_variance(
        e_hist,
        cfg.covariance_half_life_days,
        max(20, min(cfg.covariance_min_observations, len(e_hist))),
        cfg.specific_variance_floor_daily,
    ).fillna(cfg.specific_variance_floor_daily)

    B_stock = pd.concat([X, Z], axis=1)
    B_basket = A.T @ B_stock
    Omega = B_basket.to_numpy(dtype=float) @ F @ B_basket.to_numpy(dtype=float).T
    Omega += A.T.to_numpy(dtype=float) @ np.diag(specific_variance.reindex(A.index).to_numpy(dtype=float)) @ A.to_numpy(dtype=float)
    Omega, psd_diag_omega = nearest_psd(Omega, cfg.eigenvalue_floor)
    Omega += cfg.covariance_ridge * np.eye(Omega.shape[0])
    Omega_df = pd.DataFrame(Omega, index=themes, columns=themes)
    eig = np.linalg.eigvalsh(Omega)
    diagnostics = pd.Series(
        {
            "asof_date": asof,
            "n_securities": len(securities),
            "n_themes": len(themes),
            "n_factors": len(factor_cols),
            "factor_history_rows": len(f_hist),
            "theme_history_rows": len(g_hist),
            "specific_residual_history_rows": len(e_hist),
            "omega_min_eigenvalue": float(eig.min()),
            "omega_condition_number": float(np.linalg.cond(Omega)),
            "F_min_eigenvalue_before_floor": psd_diag_f["min_eigenvalue_before_floor"],
            "Omega_min_eigenvalue_before_floor": psd_diag_omega["min_eigenvalue_before_floor"],
            "covariance_ridge": cfg.covariance_ridge,
        },
        name="value",
    )
    return {
        "F_aug": F_aug,
        "Omega": Omega_df,
        "specific_variance": specific_variance,
        "basket_factor_exposure": B_basket,
        "covariance_diagnostics": diagnostics,
    }


## アロケーション手法


In [ ]:
def solve_leeliu_unconstrained(
    A_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Omega: pd.DataFrame,
    theme_importance_s: pd.Series,
    cfg: LeeLiuUnconstrainedConfig,
) -> Dict[str, Any]:
    themes = Omega.index.intersection(Omega.columns).intersection(A_t.columns).intersection(Z_t.columns).intersection(theme_importance_s.index)
    if len(themes) == 0:
        raise ValueError("共通テーマ集合が空です。")
    securities = A_t.index.intersection(Z_t.index)
    A = A_t.loc[securities, themes].astype(float)
    Z = Z_t.loc[securities, themes].astype(float)
    Q = Omega.loc[themes, themes].astype(float)
    s = theme_importance_s.reindex(themes).fillna(0.0).astype(float)

    H = A.T @ Z
    a = (H @ s).rename("theme_score_exposure")
    Qmat = 0.5 * (Q.to_numpy(dtype=float) + Q.to_numpy(dtype=float).T)
    Qinv = np.linalg.pinv(Qmat, rcond=cfg.pinv_rcond)
    C = np.column_stack([np.ones(len(themes)), a.to_numpy(dtype=float)])
    dvec = np.array([1.0, cfg.target_score_S_T], dtype=float)
    middle = np.linalg.pinv(C.T @ Qinv @ C, rcond=cfg.pinv_rcond)
    theta = Qinv @ C @ middle @ dvec
    theta_s = pd.Series(theta, index=themes, name="theta_unconstrained")
    diagnostics = pd.Series(
        {
            "theme_count": len(themes),
            "security_count": len(securities),
            "target_score_S_T": cfg.target_score_S_T,
            "budget_sum": float(theta_s.sum()),
            "budget_error": float(theta_s.sum() - 1.0),
            "realized_score": float(theta_s @ a),
            "score_error": float(theta_s @ a - cfg.target_score_S_T),
            "predicted_variance": float(theta @ Qmat @ theta),
            "omega_condition_number": float(np.linalg.cond(Qmat)),
        },
        name="value",
    )
    return {
        "theta_unconstrained": theta_s,
        "theme_score_exposure": a,
        "H": H,
        "allocation_diagnostics": diagnostics,
    }


In [ ]:
def _theme_eligibility_series(themes: pd.Index, theme_eligibility: Optional[pd.Series]) -> pd.Series:
    if theme_eligibility is None:
        return pd.Series(True, index=themes)
    elig = theme_eligibility.reindex(themes)
    if elig.dtype == bool:
        return elig.fillna(False).astype(bool)
    return elig.astype(str).str.strip().str.lower().isin({"true", "t", "1", "yes", "y"})


def _base_constraints(A: pd.DataFrame, eligible: pd.Series, cfg: ConstrainedAllocationConfig):
    K = A.shape[1]
    bounds = [(0.0, cfg.theme_cap if bool(eligible.loc[t]) else 0.0) for t in A.columns]
    A_ub = [A.to_numpy(dtype=float), np.ones((1, K))]
    b_ub = [np.full(A.shape[0], cfg.stock_cap), np.array([1.0])]
    return np.vstack(A_ub), np.concatenate(b_ub), bounds


def _lp_max_direction(direction: np.ndarray, A_ub: np.ndarray, b_ub: np.ndarray, bounds, cfg: ConstrainedAllocationConfig):
    A_eq = None if cfg.cash_allowed else np.ones((1, len(direction)))
    b_eq = None if cfg.cash_allowed else np.array([1.0])
    res = linprog(-direction, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        return None, None, res.message
    return float(direction @ res.x), res.x, res.message


def optimize_constrained_allocation(
    A_t: pd.DataFrame,
    Omega: pd.DataFrame,
    theme_score_exposure: pd.Series,
    previous_stock_weights: pd.Series,
    theme_eligibility: Optional[pd.Series],
    cfg: ConstrainedAllocationConfig,
) -> Dict[str, Any]:
    cfg.validate()
    themes = Omega.index.intersection(Omega.columns).intersection(A_t.columns).intersection(theme_score_exposure.index)
    if len(themes) == 0:
        raise ValueError("共通テーマ集合が空です。")
    A = A_t.loc[:, themes].astype(float)
    Q = Omega.loc[themes, themes].astype(float)
    a = theme_score_exposure.reindex(themes).fillna(0.0).astype(float)
    eligible = _theme_eligibility_series(themes, theme_eligibility)
    prev = previous_stock_weights.reindex(A.index).fillna(0.0).astype(float)

    A_ub, b_ub, bounds = _base_constraints(A, eligible, cfg)
    max_score, x_max, lp_message = _lp_max_direction(a.to_numpy(dtype=float), A_ub, b_ub, bounds, cfg)
    if max_score is None:
        raise RuntimeError(f"基本制約が実行不能です: {lp_message}")
    target = cfg.theme_target_fraction * max_score
    if max_score <= 1e-14 and cfg.cash_allowed:
        theta = pd.Series(0.0, index=themes, name="theta_constrained")
        stock_weights = A @ theta
        return {
            "theta_constrained": theta,
            "stock_weights": stock_weights.rename("stock_weight"),
            "cash_weight": 1.0,
            "turnover": 0.5 * float(np.abs(stock_weights - prev).sum()),
            "constraint_diagnostics": pd.Series({"status": "all_cash_nonpositive_max_score", "max_score": max_score, "target_score": target}, name="value"),
            "optimization_status": "all_cash_nonpositive_max_score",
        }

    Qmat = 0.5 * (Q.to_numpy(dtype=float) + Q.to_numpy(dtype=float).T)
    Amat = A.to_numpy(dtype=float)
    prev_vec = prev.to_numpy(dtype=float)
    avec = a.to_numpy(dtype=float)

    def objective(theta: np.ndarray) -> float:
        risk = 0.5 * float(theta @ Qmat @ theta)
        turnover_penalty = cfg.turnover_penalty_lambda * float(np.sum(np.abs(Amat @ theta - prev_vec)))
        return risk + turnover_penalty

    constraints = [LinearConstraint(A_ub, -np.inf, b_ub)]
    constraints.append(LinearConstraint(avec[None, :], [target], [target]))
    if not cfg.cash_allowed:
        constraints.append(LinearConstraint(np.ones((1, len(themes))), [1.0], [1.0]))

    x0 = np.zeros(len(themes)) if x_max is None else np.asarray(x_max, dtype=float)
    if max_score > 1e-14:
        x0 = x0 * (target / max_score)
    res = minimize(
        objective,
        x0=np.clip(x0, [b[0] for b in bounds], [b[1] for b in bounds]),
        method="SLSQP",
        bounds=Bounds([b[0] for b in bounds], [b[1] for b in bounds]),
        constraints=constraints,
        options={"maxiter": cfg.maxiter, "ftol": cfg.ftol, "disp": False},
    )
    theta_vec = res.x if res.success else x0
    theta_vec[np.abs(theta_vec) < 1e-12] = 0.0
    theta = pd.Series(theta_vec, index=themes, name="theta_constrained")
    stock_weights = (A @ theta).rename("stock_weight")
    cash_weight = max(0.0, 1.0 - float(theta.sum())) if cfg.cash_allowed else 0.0
    turnover = 0.5 * float(np.abs(stock_weights - prev).sum())
    violations = pd.Series(
        {
            "budget_plus_cash": float(theta.sum() + cash_weight),
            "budget_error": float(theta.sum() + cash_weight - 1.0),
            "realized_score": float(theta @ a),
            "score_error": float(theta @ a - target),
            "max_theme_weight": float(theta.max()) if len(theta) else np.nan,
            "max_stock_weight": float(stock_weights.max()) if len(stock_weights) else np.nan,
            "min_theta": float(theta.min()) if len(theta) else np.nan,
            "cash_weight": cash_weight,
            "turnover": turnover,
            "predicted_variance": float(theta_vec @ Qmat @ theta_vec),
            "max_score": max_score,
            "target_score": target,
            "optimizer_success": bool(res.success),
        },
        name="value",
    )
    status = "optimal" if res.success else f"feasible_lp_fallback:{res.message}"
    return {
        "theta_constrained": theta,
        "stock_weights": stock_weights,
        "cash_weight": cash_weight,
        "turnover": turnover,
        "constraint_diagnostics": violations,
        "optimization_status": status,
        "optimizer_result": res,
    }


In [ ]:
def compute_theme_score_exposure(A_t: pd.DataFrame, Z_t: pd.DataFrame, theme_importance_s: pd.Series) -> pd.Series:
    themes = A_t.columns.intersection(Z_t.columns).intersection(theme_importance_s.index)
    securities = A_t.index.intersection(Z_t.index)
    H = A_t.loc[securities, themes].T @ Z_t.loc[securities, themes]
    return (H @ theme_importance_s.reindex(themes).fillna(0.0)).rename("theme_score_exposure")


def solve_direct_alpha_allocation_with_constraints(
    A_t: pd.DataFrame,
    Omega: pd.DataFrame,
    mu_basket: pd.Series,
    previous_stock_weights: pd.Series,
    theme_eligibility: Optional[pd.Series],
    cfg: BacktestConfig,
) -> Dict[str, Any]:
    cfg.validate()
    themes = Omega.index.intersection(Omega.columns).intersection(A_t.columns).intersection(mu_basket.index)
    if len(themes) == 0:
        raise ValueError("共通テーマ集合が空です。")
    A = A_t.loc[:, themes].astype(float)
    Q = Omega.loc[themes, themes].astype(float)
    mu = mu_basket.reindex(themes).fillna(0.0).astype(float)
    eligible = _theme_eligibility_series(themes, theme_eligibility)
    prev = previous_stock_weights.reindex(A.index).fillna(0.0).astype(float)

    Qmat = 0.5 * (Q.to_numpy(dtype=float) + Q.to_numpy(dtype=float).T)
    Amat = A.to_numpy(dtype=float)
    prev_vec = prev.to_numpy(dtype=float)
    mu_vec = mu.to_numpy(dtype=float)

    def objective(theta: np.ndarray) -> float:
        utility = float(theta @ mu_vec) - 0.5 * cfg.risk_aversion_gamma * float(theta @ Qmat @ theta)
        turnover_penalty = cfg.turnover_penalty_lambda * float(np.sum(np.abs(Amat @ theta - prev_vec)))
        return -utility + turnover_penalty

    K = len(themes)
    upper = np.array([cfg.theme_cap if bool(eligible.loc[t]) else 0.0 for t in themes], dtype=float)
    constraints = [LinearConstraint(Amat, -np.inf, np.full(A.shape[0], cfg.stock_cap))]
    if cfg.allow_cash:
        constraints.append(LinearConstraint(np.ones((1, K)), -np.inf, [1.0]))
    else:
        constraints.append(LinearConstraint(np.ones((1, K)), [1.0], [1.0]))
    bounds = Bounds(np.zeros(K), upper)
    active = upper > 0
    x0 = np.zeros(K)
    if active.any():
        x0[active] = min(1.0 / active.sum(), cfg.theme_cap)
        if cfg.allow_cash and x0.sum() > 1.0:
            x0 = x0 / x0.sum()
    res = minimize(
        objective,
        x0=np.clip(x0, bounds.lb, bounds.ub),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": cfg.maxiter, "ftol": cfg.ftol, "disp": False},
    )
    theta_vec = res.x if res.success else x0
    theta_vec[np.abs(theta_vec) < 1e-12] = 0.0
    theta = pd.Series(theta_vec, index=themes, name="theta_alpha")
    stock_weights = (A @ theta).rename("stock_weight_alpha")
    cash_weight = max(0.0, 1.0 - float(theta.sum())) if cfg.allow_cash else 0.0
    diagnostics = pd.Series(
        {
            "status": "optimal" if res.success else f"fallback_initial:{res.message}",
            "optimizer_success": bool(res.success),
            "optimizer_message": str(res.message),
            "expected_alpha": float(theta @ mu),
            "predicted_variance_daily": float(theta_vec @ Qmat @ theta_vec),
            "budget_plus_cash": float(theta.sum() + cash_weight),
            "budget_error": float(theta.sum() + cash_weight - 1.0),
            "cash_weight": cash_weight,
            "max_theme_weight": float(theta.max()) if len(theta) else np.nan,
            "max_stock_weight": float(stock_weights.max()) if len(stock_weights) else np.nan,
            "min_theta": float(theta.min()) if len(theta) else np.nan,
            "turnover_objective_input": 0.5 * float(np.abs(stock_weights - prev).sum()),
        },
        name="value",
    )
    return {
        "theta_alpha": theta,
        "stock_weights_alpha": stock_weights,
        "cash_weight_alpha": cash_weight,
        "alpha_diagnostics": diagnostics,
        "optimizer_result": res,
    }


## バックテストループ


In [ ]:
def make_rebalance_dates(index: pd.DatetimeIndex, cfg: BacktestConfig) -> pd.DatetimeIndex:
    dates = pd.DatetimeIndex(index).sort_values()
    if cfg.start_date is not None:
        dates = dates[dates >= _to_naive_timestamp(cfg.start_date)]
    if cfg.end_date is not None:
        dates = dates[dates <= _to_naive_timestamp(cfg.end_date)]
    s = pd.Series(index=dates, data=dates)
    if cfg.rebalance_frequency == "weekly":
        out = s.groupby(dates.to_period(cfg.weekly_anchor)).last().dropna()
    else:
        out = s.groupby(dates.to_period("M")).last().dropna()
    return pd.DatetimeIndex(out.to_numpy())


def _warmup_days(cfg: BacktestConfig) -> int:
    return max(
        cfg.lookback_days,
        cfg.return_estimation_lookback_days,
        cfg.momentum_lookback_days + cfg.skip_recent_days + cfg.theme_exposure_lag_days,
        cfg.covariance_min_observations,
    )


def rebalance_dates_after_warmup(stock_index: pd.DatetimeIndex, cfg: BacktestConfig) -> pd.DatetimeIndex:
    all_rebalance = make_rebalance_dates(stock_index, cfg)
    if len(stock_index) == 0:
        return pd.DatetimeIndex([])
    warmup_pos = min(_warmup_days(cfg), len(stock_index) - 1)
    warmup_date = pd.Timestamp(stock_index[warmup_pos])
    return all_rebalance[all_rebalance >= warmup_date]


def compute_rebalance_inputs(
    signal_date: pd.Timestamp,
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
) -> Dict[str, Any]:
    signal_date = _to_naive_timestamp(signal_date)
    cfg_signal = replace(cfg, asof_date=f"{signal_date:%Y-%m-%d}")

    available_dates = data["stock_returns"].index[data["stock_returns"].index <= signal_date]
    if len(available_dates) < cfg.lookback_days:
        raise ValueError("coherence lookback を満たす履歴が不足しています。")
    lookback_dates = available_dates[-cfg.lookback_days:]

    base_snapshot = construct_theme_exposure_snapshot(data, stores, cfg_signal)
    ols_panel = build_daily_barra_ols_panel(data, stores, cfg_signal, lookback_dates)
    coherence_result = run_coherence_detection(base_snapshot["A_t"], ols_panel["daily_barra_residuals"], cfg_signal)
    coherence = coherence_result["coherence"]

    payoff_result = estimate_theme_return_history(data, stores, cfg_signal)
    momentum_result = compute_residual_momentum_scores(payoff_result["theme_returns_g"], cfg_signal)
    importance_result = compute_long_only_theme_importance(momentum_result["momentum_scores_q"], coherence, cfg_signal)

    purified_snapshot = compute_purified_snapshot_for_date(data, stores, cfg_signal, signal_date)
    covariance_result = build_augmented_basket_covariance(
        purified_snapshot["A_t"],
        purified_snapshot["X_t"],
        purified_snapshot["Z_t"],
        payoff_result["barra_factor_returns_f"],
        payoff_result["theme_returns_g"],
        payoff_result["theme_residuals_e"],
        cfg_signal,
    )

    A_t = purified_snapshot["A_t"].reindex(columns=covariance_result["Omega"].index).fillna(0.0)
    Z_t = purified_snapshot["Z_t"].reindex(index=A_t.index, columns=A_t.columns).fillna(0.0)
    Omega = covariance_result["Omega"].reindex(index=A_t.columns, columns=A_t.columns)
    theme_importance_s = importance_result["theme_importance_s"].reindex(A_t.columns).fillna(0.0)
    theme_score_exposure = compute_theme_score_exposure(A_t, Z_t, theme_importance_s)

    diagnostics = {
        "base_snapshot": base_snapshot["diagnostics"],
        "daily_ols": ols_panel["daily_ols_diagnostics"],
        "daily_ols_errors": ols_panel["diagnostics_errors"],
        "coherence_member": coherence_result["member_diagnostics"],
        "theme_return": payoff_result["theme_return_diagnostics"],
        "theme_return_errors": payoff_result["diagnostics_errors"],
        "momentum": momentum_result["momentum_components"],
        "importance": importance_result["importance_diagnostics"],
        "covariance": covariance_result["covariance_diagnostics"],
    }
    return {
        "signal_date": signal_date,
        "cfg_signal": cfg_signal,
        "A_t": A_t,
        "Z_t": Z_t,
        "X_t": purified_snapshot["X_t"],
        "Omega": Omega,
        "theme_importance_s": theme_importance_s,
        "theme_score_exposure": theme_score_exposure,
        "coherence": coherence,
        "momentum_scores_q": momentum_result["momentum_scores_q"],
        "momentum_components": momentum_result["momentum_components"],
        "purification_r2": purified_snapshot["purification_r2"],
        "covariance_result": covariance_result,
        "payoff_result": payoff_result,
        "diagnostics": diagnostics,
    }


def allocate_for_methods(
    rebalance_inputs: Mapping[str, Any],
    previous_stock_by_method: Mapping[str, pd.Series],
    theme_eligibility: Optional[pd.Series],
    cfg: BacktestConfig,
    methods: Sequence[str],
) -> Dict[str, Dict[str, Any]]:
    A_t = rebalance_inputs["A_t"]
    Z_t = rebalance_inputs["Z_t"]
    Omega = rebalance_inputs["Omega"]
    s = rebalance_inputs["theme_importance_s"]
    a = rebalance_inputs["theme_score_exposure"]
    out: Dict[str, Dict[str, Any]] = {}
    for method in methods:
        prev = previous_stock_by_method.get(method, pd.Series(dtype=float)).reindex(A_t.index).fillna(0.0)
        try:
            if method == "lee_liu_unconstrained":
                result = solve_leeliu_unconstrained(A_t, Z_t, Omega, s, cfg)
                theta = result["theta_unconstrained"]
                cash = 0.0
                stock = (A_t @ theta).rename("stock_weight")
                diag = result["allocation_diagnostics"].copy()
                diag.loc["status"] = "optimal"
                diag.loc["cash_weight"] = cash
                diag.loc["predicted_variance_daily"] = diag.loc["predicted_variance"]
                diag.loc["max_theme_weight"] = float(theta.max())
                diag.loc["min_theme_weight"] = float(theta.min())
                diag.loc["max_stock_weight"] = float(stock.max()) if len(stock) else np.nan
                diag.loc["min_stock_weight"] = float(stock.min()) if len(stock) else np.nan
            elif method == "constrained_realworld":
                result = optimize_constrained_allocation(A_t, Omega, a, prev, theme_eligibility, cfg)
                theta = result["theta_constrained"]
                cash = float(result["cash_weight"])
                stock = result["stock_weights"].rename("stock_weight")
                diag = result["constraint_diagnostics"].copy()
                diag.loc["status"] = result["optimization_status"]
                diag.loc["predicted_variance_daily"] = diag.get("predicted_variance", np.nan)
            elif method == "direct_alpha":
                mu_basket = (cfg.kappa * a).rename("mu_basket")
                result = solve_direct_alpha_allocation_with_constraints(A_t, Omega, mu_basket, prev, theme_eligibility, cfg)
                theta = result["theta_alpha"]
                cash = float(result["cash_weight_alpha"])
                stock = result["stock_weights_alpha"].rename("stock_weight")
                diag = result["alpha_diagnostics"].copy()
                diag.loc["mu_nonzero_count"] = int((mu_basket.abs() > 0).sum())
            else:
                raise ValueError(f"Unknown allocation method: {method}")
            reconstruction = (A_t @ theta).reindex(stock.index).fillna(0.0)
            diag.loc["stock_weight_reconstruction_error"] = float(np.nanmax(np.abs(reconstruction - stock))) if len(stock) else np.nan
            diag.loc["theme_count"] = len(theta)
            out[method] = {"theta": theta, "cash": cash, "stock": stock, "diagnostics": diag, "status": str(diag.loc["status"])}
        except Exception as exc:
            out[method] = {
                "theta": pd.Series(dtype=float),
                "cash": np.nan,
                "stock": pd.Series(dtype=float),
                "diagnostics": pd.Series({"status": f"allocation_error:{type(exc).__name__}", "message": str(exc)}, name="value"),
                "status": f"allocation_error:{type(exc).__name__}",
            }
    return out


def run_backtest_allocation_methods(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    methods: Sequence[str],
    cash_returns: pd.Series,
    theme_eligibility: Optional[pd.Series],
) -> Dict[str, Any]:
    cfg.validate()
    stock_returns = data["stock_returns"]
    all_dates = stock_returns.index
    rebalance_dates = rebalance_dates_after_warmup(all_dates, cfg)
    if len(rebalance_dates) == 0:
        raise ValueError("warmup後のrebalance dateがありません。期間設定を確認してください。")
    sim_start = rebalance_dates.min()
    if cfg.start_date is not None:
        sim_start = max(sim_start, _to_naive_timestamp(cfg.start_date))
    sim_end = all_dates.max() if cfg.end_date is None else min(all_dates.max(), _to_naive_timestamp(cfg.end_date))
    sim_dates = all_dates[(all_dates >= sim_start) & (all_dates <= sim_end)]
    methods = list(methods)

    current_w = {m: pd.Series(0.0, index=stock_returns.columns, dtype=float) for m in methods}
    current_cash = {m: 1.0 for m in methods}
    pending_targets: Dict[pd.Timestamp, Dict[str, Dict[str, Any]]] = {}

    gross = pd.DataFrame(0.0, index=sim_dates, columns=methods, dtype=float)
    net = gross.copy()
    turnover = gross.copy()
    transaction_cost = gross.copy()
    cash_weight = pd.DataFrame(1.0, index=sim_dates, columns=methods, dtype=float)

    target_theme_rows = []
    target_stock_rows = []
    opt_logs = []
    coherence_z_rows = []
    coherence_p_rows = []
    coherence_q_rows = []
    momentum_rows = []
    importance_rows = []
    pur_r2_rows = []
    cov_diag_rows = []
    predicted_vol_rows = []

    rebalance_set = set(pd.Timestamp(x) for x in rebalance_dates)
    for d in sim_dates:
        d = pd.Timestamp(d)
        day_targets = pending_targets.pop(d, {})
        daily_tc = {m: 0.0 for m in methods}
        for method, target in day_targets.items():
            target_stock = target["stock"].reindex(stock_returns.columns).fillna(0.0)
            target_cash = float(target["cash"])
            one_way = 0.5 * float(np.abs(target_stock - current_w[method]).sum())
            turnover.loc[d, method] = one_way
            daily_tc[method] = one_way * cfg.transaction_cost_bps_one_way / 10000.0
            transaction_cost.loc[d, method] = daily_tc[method]
            current_w[method] = target_stock
            current_cash[method] = target_cash

        rday = stock_returns.loc[d].fillna(0.0)
        cash_r = float(cash_returns.reindex([d]).fillna(cfg.cash_return_default_daily).iloc[0])
        for method in methods:
            gross.loc[d, method] = float(current_w[method] @ rday) + current_cash[method] * cash_r
            net.loc[d, method] = gross.loc[d, method] - daily_tc[method]
            stock_values = current_w[method] * (1.0 + rday)
            cash_value = current_cash[method] * (1.0 + cash_r)
            total = float(stock_values.sum() + cash_value)
            if abs(total) > 1e-14:
                current_w[method] = stock_values / total
                current_cash[method] = cash_value / total
            cash_weight.loc[d, method] = current_cash[method]

        if d not in rebalance_set:
            continue

        signal_pos = all_dates.searchsorted(d)
        execution_pos = signal_pos + cfg.execution_lag_days
        if execution_pos >= len(all_dates):
            opt_logs.append({"signal_date": d, "status": "skip_no_execution_date"})
            continue
        execution_date = pd.Timestamp(all_dates[execution_pos])

        try:
            rebalance_inputs = compute_rebalance_inputs(d, data, stores, cfg)
        except Exception as exc:
            for method in methods:
                opt_logs.append({"signal_date": d, "execution_date": execution_date, "method": method, "status": f"signal_error:{type(exc).__name__}", "message": str(exc)})
            continue

        coherence = rebalance_inputs["coherence"]
        coherence_z_rows.append(coherence["z"].rename(d))
        coherence_p_rows.append(coherence["p"].rename(d))
        coherence_q_rows.append(coherence["q"].rename(d))
        momentum_rows.append(rebalance_inputs["momentum_scores_q"].rename(d))
        importance_rows.append(rebalance_inputs["theme_importance_s"].rename(d))
        if "adjusted_r2" in rebalance_inputs["purification_r2"].columns:
            pur_r2_rows.append(rebalance_inputs["purification_r2"]["adjusted_r2"].rename(d))
        cov_diag = rebalance_inputs["covariance_result"]["covariance_diagnostics"].copy()
        cov_diag.name = d
        cov_diag_rows.append(cov_diag)

        allocations = allocate_for_methods(
            rebalance_inputs,
            current_w,
            theme_eligibility,
            cfg,
            methods,
        )
        pending_targets.setdefault(execution_date, {})
        pred_vol_row = pd.Series(index=methods, dtype=float, name=d)
        for method, alloc in allocations.items():
            diag = alloc["diagnostics"].copy()
            diag_dict = diag.to_dict()
            diag_dict.update({"signal_date": d, "execution_date": execution_date, "method": method})
            opt_logs.append(diag_dict)
            if alloc["stock"].empty:
                continue
            pending_targets[execution_date][method] = alloc
            theta_row = alloc["theta"].rename((execution_date, method))
            stock_row = alloc["stock"].rename((execution_date, method))
            target_theme_rows.append(theta_row)
            target_stock_rows.append(stock_row)
            var_daily = pd.to_numeric(pd.Series([diag.get("predicted_variance_daily", np.nan)]), errors="coerce").iloc[0]
            pred_vol_row.loc[method] = math.sqrt(max(float(var_daily), 0.0) * cfg.annualization) if np.isfinite(var_daily) else np.nan
        predicted_vol_rows.append(pred_vol_row)
        if cfg.verbose:
            print(f"{d.date()} -> {execution_date.date()} signals computed")

    target_theme_weights = pd.DataFrame(target_theme_rows)
    target_stock_weights = pd.DataFrame(target_stock_rows)
    if len(target_theme_weights):
        target_theme_weights.index = pd.MultiIndex.from_tuples(target_theme_weights.index, names=["execution_date", "method"])
    if len(target_stock_weights):
        target_stock_weights.index = pd.MultiIndex.from_tuples(target_stock_weights.index, names=["execution_date", "method"])

    diagnostics_by_rebalance = {
        "coherence_z": pd.DataFrame(coherence_z_rows).sort_index() if coherence_z_rows else pd.DataFrame(),
        "coherence_p": pd.DataFrame(coherence_p_rows).sort_index() if coherence_p_rows else pd.DataFrame(),
        "coherence_q": pd.DataFrame(coherence_q_rows).sort_index() if coherence_q_rows else pd.DataFrame(),
        "momentum_score": pd.DataFrame(momentum_rows).sort_index() if momentum_rows else pd.DataFrame(),
        "theme_importance": pd.DataFrame(importance_rows).sort_index() if importance_rows else pd.DataFrame(),
        "purification_adjusted_r2": pd.DataFrame(pur_r2_rows).sort_index() if pur_r2_rows else pd.DataFrame(),
        "covariance_diagnostics": pd.DataFrame(cov_diag_rows).sort_index() if cov_diag_rows else pd.DataFrame(),
        "predicted_volatility": pd.DataFrame(predicted_vol_rows).sort_index() if predicted_vol_rows else pd.DataFrame(),
    }

    monthly_returns = (1.0 + net).resample("ME").prod() - 1.0
    metrics = performance_metrics(net, gross, turnover, transaction_cost, cfg.annualization)
    return {
        "rebalance_dates": rebalance_dates,
        "daily_returns_gross": gross,
        "daily_returns_net": net,
        "daily_turnover": turnover,
        "daily_transaction_cost": transaction_cost,
        "daily_cash_weight": cash_weight,
        "target_theme_weights": target_theme_weights,
        "target_stock_weights": target_stock_weights,
        "optimization_log": pd.DataFrame(opt_logs),
        "diagnostics_by_rebalance": diagnostics_by_rebalance,
        "monthly_returns_net": monthly_returns,
        "metrics": metrics,
    }


## 事前計算pkl生成・ロード

バックテスト前に日次Barra OLS、純化snapshot、純化テーマ・リターン履歴をpkl化する。バックテスト実行時はこれらをロードし、同じシグナル・リスク・配分ロジックへ渡す。


In [ ]:
import hashlib
import json
from datetime import datetime, timezone

PRECOMPUTE_FILES = {
    "daily_barra_ols": "daily_barra_ols.pkl",
    "purified_snapshots": "purified_snapshots.pkl",
    "theme_return_history": "theme_return_history.pkl",
    "metadata": "precompute_metadata.pkl",
}

PRECOMPUTE_SIGNATURE_FIELDS = [
    "precompute_version",
    "theme_exposure_mode",
    "min_basket_coverage",
    "pure_exposure_min_norm",
    "pinv_rcond",
    "theme_return_ridge_alpha",
    "theme_exposure_lag_days",
    "x_dir",
    "mapping_path",
]


def _date_list(values: Sequence[pd.Timestamp]) -> list[pd.Timestamp]:
    return [pd.Timestamp(x).normalize() for x in pd.DatetimeIndex(values).sort_values().unique()]


def precompute_config_payload(cfg: BacktestConfig) -> Dict[str, Any]:
    return {field: getattr(cfg, field) for field in PRECOMPUTE_SIGNATURE_FIELDS}


def precompute_config_signature(cfg: BacktestConfig) -> str:
    payload = precompute_config_payload(cfg)
    raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def required_precompute_dates(data: Mapping[str, Any], cfg: BacktestConfig) -> Dict[str, Any]:
    stock_index = pd.DatetimeIndex(data["stock_returns"].index).sort_values()
    rebalance_dates = rebalance_dates_after_warmup(stock_index, cfg)
    daily_ols_dates: set[pd.Timestamp] = set()
    theme_return_dates: set[pd.Timestamp] = set()
    snapshot_dates: set[pd.Timestamp] = set(pd.Timestamp(x).normalize() for x in rebalance_dates)

    for signal_date in rebalance_dates:
        signal_date = pd.Timestamp(signal_date).normalize()
        available = stock_index[stock_index <= signal_date]
        if len(available) < cfg.lookback_days:
            continue
        daily_ols_dates.update(pd.Timestamp(x).normalize() for x in available[-cfg.lookback_days:])

        return_window = available[-cfg.return_estimation_lookback_days:]
        for d in return_window:
            d = pd.Timestamp(d).normalize()
            try:
                snapshot_date = previous_trading_date(stock_index, d, cfg.theme_exposure_lag_days)
            except KeyError:
                continue
            theme_return_dates.add(d)
            daily_ols_dates.add(d)
            snapshot_dates.add(pd.Timestamp(snapshot_date).normalize())

    return {
        "rebalance_dates": _date_list(rebalance_dates),
        "daily_ols_dates": _date_list(daily_ols_dates),
        "theme_return_dates": _date_list(theme_return_dates),
        "snapshot_dates": _date_list(snapshot_dates),
    }


def _series_rows_to_frame(rows: list[pd.Series]) -> pd.DataFrame:
    return pd.DataFrame(rows).sort_index() if rows else pd.DataFrame()


def build_daily_barra_ols_precomputed(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    dates: Sequence[pd.Timestamp],
) -> Tuple[Dict[pd.Timestamp, Dict[str, Any]], Dict[str, Any]]:
    by_date: Dict[pd.Timestamp, Dict[str, Any]] = {}
    residual_rows = []
    fitted_rows = []
    factor_rows = []
    diag_rows = []
    errors = []
    for d in _date_list(dates):
        try:
            result = run_daily_barra_ols(data, stores, cfg, d)
        except Exception as exc:
            errors.append({"date": d, "status": f"error:{type(exc).__name__}", "message": str(exc)})
            continue
        by_date[d] = result
        residual_rows.append(result["residuals_u"])
        fitted_rows.append(result["fitted_returns"])
        factor_rows.append(result["factor_returns_f"])
        diag_rows.append(result["diagnostics"])
    errors_df = pd.DataFrame(errors).set_index("date") if errors else pd.DataFrame(columns=["status", "message"])
    missing = pd.DatetimeIndex(dates).difference(pd.DatetimeIndex(by_date))
    if len(missing) > 0:
        raise RuntimeError(f"daily_barra_ols の事前計算に失敗した日付があります: {[str(x.date()) for x in missing[:10]]}")
    artifact = {
        "barra_factor_returns_f": _series_rows_to_frame(factor_rows),
        "daily_barra_residuals": _series_rows_to_frame(residual_rows),
        "daily_barra_fitted": _series_rows_to_frame(fitted_rows),
        "daily_ols_diagnostics": _series_rows_to_frame(diag_rows),
        "daily_ols_errors": errors_df,
    }
    return by_date, artifact


def compute_purified_snapshot_for_precompute(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    asof_date: pd.Timestamp,
) -> Dict[str, Any]:
    cfg_for_date = replace(cfg, asof_date=f"{_to_naive_timestamp(asof_date):%Y-%m-%d}")
    base_snapshot = construct_theme_exposure_snapshot(data, stores, cfg_for_date)
    purified = purify_theme_exposures(base_snapshot["X"], base_snapshot["X_M_t"], cfg_for_date)
    Z_t = purified["Z_t"]
    A_t = base_snapshot["A_t"].reindex(index=Z_t.index, columns=Z_t.columns).fillna(0.0)
    return {
        "asof_date": base_snapshot["asof_date"],
        "base_A_t": base_snapshot["A_t"],
        "base_X_M_t": base_snapshot["X_M_t"],
        "A_t": A_t,
        "X_t": purified["X_t"],
        "X_M_t": purified["X_M_t"],
        "Z_t": Z_t,
        "purification_r2": purified["purification_r2"],
        "pure_norm": purified["pure_norm"],
        "diagnostics": {**base_snapshot["diagnostics"], "purification_model": purified["model_diagnostics"]},
    }


def build_purified_snapshots_precomputed(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    dates: Sequence[pd.Timestamp],
) -> Dict[pd.Timestamp, Dict[str, Any]]:
    snapshots: Dict[pd.Timestamp, Dict[str, Any]] = {}
    errors = []
    for d in _date_list(dates):
        try:
            snapshots[d] = compute_purified_snapshot_for_precompute(data, stores, cfg, d)
        except Exception as exc:
            errors.append({"date": d, "status": f"error:{type(exc).__name__}", "message": str(exc)})
    if errors:
        preview = [f"{x['date'].date()}:{x['status']}" for x in errors[:10]]
        raise RuntimeError(f"purified_snapshots の事前計算に失敗した日付があります: {preview}")
    return snapshots


def estimate_theme_return_for_date_from_precomputed(
    data: Mapping[str, Any],
    cfg: BacktestConfig,
    return_date: pd.Timestamp,
    daily_ols_by_date: Mapping[pd.Timestamp, Dict[str, Any]],
    purified_snapshots: Mapping[pd.Timestamp, Dict[str, Any]],
) -> Dict[str, Any]:
    d = _to_naive_timestamp(return_date)
    snapshot_date = previous_trading_date(data["stock_returns"].index, d, cfg.theme_exposure_lag_days)
    if d not in daily_ols_by_date:
        raise KeyError(f"daily_barra_ols precompute missing date={d.date()}")
    if snapshot_date not in purified_snapshots:
        raise KeyError(f"purified_snapshot precompute missing date={snapshot_date.date()}")
    barra = daily_ols_by_date[d]
    snap = purified_snapshots[snapshot_date]

    u = barra["residuals_u"]
    Z = snap["Z_t"]
    common = Z.index.intersection(u.index)
    Z = Z.loc[common]
    u = u.reindex(common)
    valid = u.notna() & Z.notna().all(axis=1)
    Z = Z.loc[valid]
    u = u.loc[valid]
    if Z.empty or len(u) == 0:
        raise ValueError(f"{d.date()} のテーマリターン推定に使える銘柄がありません。")

    Zmat = Z.to_numpy(dtype=float)
    uvec = u.to_numpy(dtype=float)
    K = Z.shape[1]
    lhs = Zmat.T @ Zmat + cfg.theme_return_ridge_alpha * np.eye(K)
    rhs = Zmat.T @ uvec
    g = np.linalg.pinv(lhs, rcond=cfg.pinv_rcond) @ rhs
    fitted_theme = Zmat @ g
    e = uvec - fitted_theme

    denom = np.sum(Zmat * Zmat, axis=0)
    g_uni = np.divide(rhs, denom, out=np.full(K, np.nan), where=denom > 0)
    z_norm = pd.Series(np.sqrt(denom), index=Z.columns, name=d)

    diag = pd.Series(
        {
            "return_date": d,
            "snapshot_date_for_Z": snapshot_date,
            "n_securities": len(u),
            "n_themes": K,
            "z_rank": int(np.linalg.matrix_rank(Zmat)),
            "ridge_alpha": cfg.theme_return_ridge_alpha,
            "max_abs_z_norm_error": float(np.nanmax(np.abs(z_norm - 1.0))) if len(z_norm) else np.nan,
            "barra_ols_n_securities": barra["diagnostics"].loc["ols_n_securities"],
            "barra_ols_rank": barra["diagnostics"].loc["ols_rank"],
        },
        name=d,
    )
    return {
        "date": d,
        "snapshot_date": snapshot_date,
        "theme_returns_g": pd.Series(g, index=Z.columns, name=d),
        "univariate_theme_returns_g": pd.Series(g_uni, index=Z.columns, name=d),
        "theme_residuals_e": pd.Series(e, index=Z.index, name=d),
        "barra_residuals_u": u.rename(d),
        "barra_factor_returns_f": barra["factor_returns_f"],
        "z_norm": z_norm,
        "diagnostics": diag,
    }


def build_theme_return_history_precomputed(
    data: Mapping[str, Any],
    cfg: BacktestConfig,
    dates: Sequence[pd.Timestamp],
    daily_ols_by_date: Mapping[pd.Timestamp, Dict[str, Any]],
    purified_snapshots: Mapping[pd.Timestamp, Dict[str, Any]],
) -> Dict[str, Any]:
    rows_g = []
    rows_g_uni = []
    rows_e = []
    rows_u = []
    rows_f = []
    rows_diag = []
    rows_norm = []
    errors = []
    for d in _date_list(dates):
        try:
            result = estimate_theme_return_for_date_from_precomputed(data, cfg, d, daily_ols_by_date, purified_snapshots)
        except Exception as exc:
            errors.append({"date": d, "status": f"error:{type(exc).__name__}", "message": str(exc)})
            continue
        rows_g.append(result["theme_returns_g"])
        rows_g_uni.append(result["univariate_theme_returns_g"])
        rows_e.append(result["theme_residuals_e"])
        rows_u.append(result["barra_residuals_u"])
        rows_f.append(result["barra_factor_returns_f"])
        rows_diag.append(result["diagnostics"])
        rows_norm.append(result["z_norm"])
    errors_df = pd.DataFrame(errors).set_index("date") if errors else pd.DataFrame(columns=["status", "message"])
    missing = pd.DatetimeIndex(dates).difference(pd.DatetimeIndex([x.name for x in rows_g]))
    if len(missing) > 0:
        raise RuntimeError(f"theme_return_history の事前計算に失敗した日付があります: {[str(x.date()) for x in missing[:10]]}")
    return {
        "theme_returns_g": _series_rows_to_frame(rows_g),
        "univariate_theme_returns_g": _series_rows_to_frame(rows_g_uni),
        "theme_residuals_e": _series_rows_to_frame(rows_e),
        "barra_residuals_u": _series_rows_to_frame(rows_u),
        "barra_factor_returns_f": _series_rows_to_frame(rows_f),
        "z_norms": _series_rows_to_frame(rows_norm),
        "theme_return_diagnostics": _series_rows_to_frame(rows_diag),
        "diagnostics_errors": errors_df,
    }


def precompute_item10_artifacts(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    precompute_dir: str | Path | None = None,
) -> Dict[str, Any]:
    cfg.validate()
    out_dir = Path(precompute_dir or cfg.precompute_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    required = required_precompute_dates(data, cfg)
    daily_ols_by_date, daily_ols_artifact = build_daily_barra_ols_precomputed(data, stores, cfg, required["daily_ols_dates"])
    purified_snapshots = build_purified_snapshots_precomputed(data, stores, cfg, required["snapshot_dates"])
    theme_return_history = build_theme_return_history_precomputed(
        data,
        cfg,
        required["theme_return_dates"],
        daily_ols_by_date,
        purified_snapshots,
    )
    metadata = {
        "cache_version": cfg.precompute_version,
        "config_payload": precompute_config_payload(cfg),
        "config_signature": precompute_config_signature(cfg),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "precompute_dir": str(out_dir),
        "x_dir": cfg.x_dir,
        "mapping_path": cfg.mapping_path,
        "target_start_date": str(pd.DatetimeIndex(data["stock_returns"].index).min().date()),
        "target_end_date": str(pd.DatetimeIndex(data["stock_returns"].index).max().date()),
        "required_dates": {k: [str(pd.Timestamp(x).date()) for x in v] for k, v in required.items()},
        "artifact_files": PRECOMPUTE_FILES,
        "x_mapping_exact_match_checked": True,
    }
    pd.to_pickle(daily_ols_artifact, out_dir / PRECOMPUTE_FILES["daily_barra_ols"])
    pd.to_pickle(purified_snapshots, out_dir / PRECOMPUTE_FILES["purified_snapshots"])
    pd.to_pickle(theme_return_history, out_dir / PRECOMPUTE_FILES["theme_return_history"])
    pd.to_pickle(metadata, out_dir / PRECOMPUTE_FILES["metadata"])
    return {
        "precompute_dir": out_dir,
        "metadata": metadata,
        "daily_barra_ols": daily_ols_artifact,
        "purified_snapshots": purified_snapshots,
        "theme_return_history": theme_return_history,
    }


def _require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"事前計算pklがありません: {path}")


def _dates_from_metadata(metadata: Mapping[str, Any], key: str) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(pd.to_datetime(metadata["required_dates"].get(key, []))).normalize()


def validate_precomputed_bundle(bundle: Mapping[str, Any], data: Mapping[str, Any], cfg: BacktestConfig) -> None:
    metadata = bundle["metadata"]
    expected_signature = precompute_config_signature(cfg)
    if metadata.get("config_signature") != expected_signature:
        raise ValueError("precompute_metadata の config_signature が現在の設定と一致しません。pklを再作成してください。")
    if metadata.get("cache_version") != cfg.precompute_version:
        raise ValueError("precompute_metadata の cache_version が現在の設定と一致しません。")

    required_now = required_precompute_dates(data, cfg)
    stored = metadata.get("required_dates", {})
    for key, dates in required_now.items():
        need = set(str(pd.Timestamp(x).date()) for x in dates)
        have = set(stored.get(key, []))
        missing = sorted(need - have)
        if missing:
            raise ValueError(f"precompute pkl に必要日付が不足しています: {key} {missing[:10]}")

    daily = bundle["daily_barra_ols"]
    daily_idx = pd.DatetimeIndex(daily["daily_barra_residuals"].index).normalize()
    missing_daily = pd.DatetimeIndex(required_now["daily_ols_dates"]).difference(daily_idx)
    if len(missing_daily) > 0:
        raise ValueError(f"daily_barra_ols.pkl に必要日付がありません: {[str(x.date()) for x in missing_daily[:10]]}")

    snapshots = bundle["purified_snapshots"]
    snapshot_keys = pd.DatetimeIndex([pd.Timestamp(x).normalize() for x in snapshots.keys()])
    missing_snapshots = pd.DatetimeIndex(required_now["snapshot_dates"]).difference(snapshot_keys)
    if len(missing_snapshots) > 0:
        raise ValueError(f"purified_snapshots.pkl に必要日付がありません: {[str(x.date()) for x in missing_snapshots[:10]]}")

    theme = bundle["theme_return_history"]
    theme_idx = pd.DatetimeIndex(theme["theme_returns_g"].index).normalize()
    missing_theme = pd.DatetimeIndex(required_now["theme_return_dates"]).difference(theme_idx)
    if len(missing_theme) > 0:
        raise ValueError(f"theme_return_history.pkl に必要日付がありません: {[str(x.date()) for x in missing_theme[:10]]}")


def load_precomputed_bundle(precompute_dir: str | Path, cfg: BacktestConfig, data: Mapping[str, Any]) -> Dict[str, Any]:
    base = Path(precompute_dir)
    paths = {name: base / filename for name, filename in PRECOMPUTE_FILES.items()}
    for p in paths.values():
        _require_file(p)
    bundle = {
        "daily_barra_ols": pd.read_pickle(paths["daily_barra_ols"]),
        "purified_snapshots": pd.read_pickle(paths["purified_snapshots"]),
        "theme_return_history": pd.read_pickle(paths["theme_return_history"]),
        "metadata": pd.read_pickle(paths["metadata"]),
        "source": "precomputed_pkl",
        "precompute_dir": base,
    }
    validate_precomputed_bundle(bundle, data, cfg)
    return bundle


## pklロード版バックテストループ

この経路ではバックテスト中に `run_daily_barra_ols()`、`compute_purified_snapshot_for_date()`、`estimate_theme_return_for_date()` を呼ばない。ロード済みpklから必要な日付とlookback windowを切り出す。


In [ ]:
def _slice_daily_ols_panel_from_precomputed(bundle: Mapping[str, Any], dates: Sequence[pd.Timestamp]) -> Dict[str, Any]:
    artifact = bundle["daily_barra_ols"]
    dates_idx = pd.DatetimeIndex(pd.to_datetime(dates)).normalize()
    missing = dates_idx.difference(pd.DatetimeIndex(artifact["daily_barra_residuals"].index).normalize())
    if len(missing) > 0:
        raise ValueError(f"daily_barra_ols.pkl にlookback日付が不足しています: {[str(x.date()) for x in missing[:10]]}")
    return {
        "daily_barra_residuals": artifact["daily_barra_residuals"].reindex(dates_idx),
        "daily_barra_fitted": artifact["daily_barra_fitted"].reindex(dates_idx),
        "barra_factor_returns_f": artifact["barra_factor_returns_f"].reindex(dates_idx),
        "daily_ols_diagnostics": artifact["daily_ols_diagnostics"].reindex(dates_idx),
        "diagnostics_errors": artifact["daily_ols_errors"],
    }


def _slice_theme_return_history_from_precomputed(bundle: Mapping[str, Any], cfg: BacktestConfig) -> Dict[str, Any]:
    artifact = bundle["theme_return_history"]
    asof = _to_naive_timestamp(cfg.asof_date)
    available = artifact["theme_returns_g"].index[artifact["theme_returns_g"].index <= asof]
    dates = pd.DatetimeIndex(available[-cfg.return_estimation_lookback_days :]).normalize()
    if len(dates) == 0:
        raise ValueError(f"theme_return_history.pkl に asof={asof.date()} 以前の履歴がありません。")
    out = {}
    for key, value in artifact.items():
        if isinstance(value, pd.DataFrame) and key != "diagnostics_errors":
            out[key] = value.reindex(dates)
        else:
            out[key] = value
    return out


def _precomputed_snapshot(bundle: Mapping[str, Any], date: pd.Timestamp) -> Dict[str, Any]:
    d = _to_naive_timestamp(date)
    snapshots = bundle["purified_snapshots"]
    if d not in snapshots:
        raise KeyError(f"purified_snapshots.pkl に date={d.date()} がありません。")
    return snapshots[d]


def compute_rebalance_inputs_from_precomputed(
    signal_date: pd.Timestamp,
    data: Mapping[str, Any],
    cfg: BacktestConfig,
    precomputed_bundle: Mapping[str, Any],
) -> Dict[str, Any]:
    signal_date = _to_naive_timestamp(signal_date)
    cfg_signal = replace(cfg, asof_date=f"{signal_date:%Y-%m-%d}")

    available_dates = data["stock_returns"].index[data["stock_returns"].index <= signal_date]
    if len(available_dates) < cfg.lookback_days:
        raise ValueError("coherence lookback を満たす履歴が不足しています。")
    lookback_dates = available_dates[-cfg.lookback_days:]

    purified_snapshot = _precomputed_snapshot(precomputed_bundle, signal_date)
    base_A_t = purified_snapshot.get("base_A_t", purified_snapshot["A_t"])
    ols_panel = _slice_daily_ols_panel_from_precomputed(precomputed_bundle, lookback_dates)
    coherence_result = run_coherence_detection(base_A_t, ols_panel["daily_barra_residuals"], cfg_signal)
    coherence = coherence_result["coherence"]

    payoff_result = _slice_theme_return_history_from_precomputed(precomputed_bundle, cfg_signal)
    momentum_result = compute_residual_momentum_scores(payoff_result["theme_returns_g"], cfg_signal)
    importance_result = compute_long_only_theme_importance(momentum_result["momentum_scores_q"], coherence, cfg_signal)

    covariance_result = build_augmented_basket_covariance(
        purified_snapshot["A_t"],
        purified_snapshot["X_t"],
        purified_snapshot["Z_t"],
        payoff_result["barra_factor_returns_f"],
        payoff_result["theme_returns_g"],
        payoff_result["theme_residuals_e"],
        cfg_signal,
    )

    A_t = purified_snapshot["A_t"].reindex(columns=covariance_result["Omega"].index).fillna(0.0)
    Z_t = purified_snapshot["Z_t"].reindex(index=A_t.index, columns=A_t.columns).fillna(0.0)
    Omega = covariance_result["Omega"].reindex(index=A_t.columns, columns=A_t.columns)
    theme_importance_s = importance_result["theme_importance_s"].reindex(A_t.columns).fillna(0.0)
    theme_score_exposure = compute_theme_score_exposure(A_t, Z_t, theme_importance_s)

    diagnostics = {
        "source": "precomputed_pkl",
        "daily_ols": ols_panel["daily_ols_diagnostics"],
        "daily_ols_errors": ols_panel["diagnostics_errors"],
        "coherence_member": coherence_result["member_diagnostics"],
        "theme_return": payoff_result["theme_return_diagnostics"],
        "theme_return_errors": payoff_result["diagnostics_errors"],
        "momentum": momentum_result["momentum_components"],
        "importance": importance_result["importance_diagnostics"],
        "covariance": covariance_result["covariance_diagnostics"],
    }
    return {
        "signal_date": signal_date,
        "cfg_signal": cfg_signal,
        "A_t": A_t,
        "Z_t": Z_t,
        "X_t": purified_snapshot["X_t"],
        "Omega": Omega,
        "theme_importance_s": theme_importance_s,
        "theme_score_exposure": theme_score_exposure,
        "coherence": coherence,
        "momentum_scores_q": momentum_result["momentum_scores_q"],
        "momentum_components": momentum_result["momentum_components"],
        "purification_r2": purified_snapshot["purification_r2"],
        "covariance_result": covariance_result,
        "payoff_result": payoff_result,
        "diagnostics": diagnostics,
    }


def run_backtest_allocation_methods_precomputed(
    data: Mapping[str, Any],
    cfg: BacktestConfig,
    methods: Sequence[str],
    cash_returns: pd.Series,
    theme_eligibility: Optional[pd.Series],
    precomputed_bundle: Mapping[str, Any],
) -> Dict[str, Any]:
    cfg.validate()
    stock_returns = data["stock_returns"]
    all_dates = stock_returns.index
    rebalance_dates = rebalance_dates_after_warmup(all_dates, cfg)
    if len(rebalance_dates) == 0:
        raise ValueError("warmup後のrebalance dateがありません。期間設定を確認してください。")
    sim_start = rebalance_dates.min()
    if cfg.start_date is not None:
        sim_start = max(sim_start, _to_naive_timestamp(cfg.start_date))
    sim_end = all_dates.max() if cfg.end_date is None else min(all_dates.max(), _to_naive_timestamp(cfg.end_date))
    sim_dates = all_dates[(all_dates >= sim_start) & (all_dates <= sim_end)]
    methods = list(methods)

    current_w = {m: pd.Series(0.0, index=stock_returns.columns, dtype=float) for m in methods}
    current_cash = {m: 1.0 for m in methods}
    pending_targets: Dict[pd.Timestamp, Dict[str, Dict[str, Any]]] = {}

    gross = pd.DataFrame(0.0, index=sim_dates, columns=methods, dtype=float)
    net = gross.copy()
    turnover = gross.copy()
    transaction_cost = gross.copy()
    cash_weight = pd.DataFrame(1.0, index=sim_dates, columns=methods, dtype=float)

    target_theme_rows = []
    target_stock_rows = []
    opt_logs = []
    coherence_z_rows = []
    coherence_p_rows = []
    coherence_q_rows = []
    momentum_rows = []
    importance_rows = []
    pur_r2_rows = []
    cov_diag_rows = []
    predicted_vol_rows = []

    rebalance_set = set(pd.Timestamp(x) for x in rebalance_dates)
    for d in sim_dates:
        d = pd.Timestamp(d)
        day_targets = pending_targets.pop(d, {})
        daily_tc = {m: 0.0 for m in methods}
        for method, target in day_targets.items():
            target_stock = target["stock"].reindex(stock_returns.columns).fillna(0.0)
            target_cash = float(target["cash"])
            one_way = 0.5 * float(np.abs(target_stock - current_w[method]).sum())
            turnover.loc[d, method] = one_way
            daily_tc[method] = one_way * cfg.transaction_cost_bps_one_way / 10000.0
            transaction_cost.loc[d, method] = daily_tc[method]
            current_w[method] = target_stock
            current_cash[method] = target_cash

        rday = stock_returns.loc[d].fillna(0.0)
        cash_r = float(cash_returns.reindex([d]).fillna(cfg.cash_return_default_daily).iloc[0])
        for method in methods:
            gross.loc[d, method] = float(current_w[method] @ rday) + current_cash[method] * cash_r
            net.loc[d, method] = gross.loc[d, method] - daily_tc[method]
            stock_values = current_w[method] * (1.0 + rday)
            cash_value = current_cash[method] * (1.0 + cash_r)
            total = float(stock_values.sum() + cash_value)
            if abs(total) > 1e-14:
                current_w[method] = stock_values / total
                current_cash[method] = cash_value / total
            cash_weight.loc[d, method] = current_cash[method]

        if d not in rebalance_set:
            continue

        signal_pos = all_dates.searchsorted(d)
        execution_pos = signal_pos + cfg.execution_lag_days
        if execution_pos >= len(all_dates):
            opt_logs.append({"signal_date": d, "status": "skip_no_execution_date", "input_source": "precomputed_pkl"})
            continue
        execution_date = pd.Timestamp(all_dates[execution_pos])

        try:
            rebalance_inputs = compute_rebalance_inputs_from_precomputed(d, data, cfg, precomputed_bundle)
        except Exception as exc:
            for method in methods:
                opt_logs.append({"signal_date": d, "execution_date": execution_date, "method": method, "status": f"signal_error:{type(exc).__name__}", "message": str(exc), "input_source": "precomputed_pkl"})
            continue

        coherence = rebalance_inputs["coherence"]
        coherence_z_rows.append(coherence["z"].rename(d))
        coherence_p_rows.append(coherence["p"].rename(d))
        coherence_q_rows.append(coherence["q"].rename(d))
        momentum_rows.append(rebalance_inputs["momentum_scores_q"].rename(d))
        importance_rows.append(rebalance_inputs["theme_importance_s"].rename(d))
        if "adjusted_r2" in rebalance_inputs["purification_r2"].columns:
            pur_r2_rows.append(rebalance_inputs["purification_r2"]["adjusted_r2"].rename(d))
        cov_diag = rebalance_inputs["covariance_result"]["covariance_diagnostics"].copy()
        cov_diag.name = d
        cov_diag_rows.append(cov_diag)

        allocations = allocate_for_methods(
            rebalance_inputs,
            current_w,
            theme_eligibility,
            cfg,
            methods,
        )
        pending_targets.setdefault(execution_date, {})
        pred_vol_row = pd.Series(index=methods, dtype=float, name=d)
        for method, alloc in allocations.items():
            diag = alloc["diagnostics"].copy()
            diag_dict = diag.to_dict()
            diag_dict.update({"signal_date": d, "execution_date": execution_date, "method": method, "input_source": "precomputed_pkl"})
            opt_logs.append(diag_dict)
            if alloc["stock"].empty:
                continue
            pending_targets[execution_date][method] = alloc
            theta_row = alloc["theta"].rename((execution_date, method))
            stock_row = alloc["stock"].rename((execution_date, method))
            target_theme_rows.append(theta_row)
            target_stock_rows.append(stock_row)
            var_daily = pd.to_numeric(pd.Series([diag.get("predicted_variance_daily", np.nan)]), errors="coerce").iloc[0]
            pred_vol_row.loc[method] = math.sqrt(max(float(var_daily), 0.0) * cfg.annualization) if np.isfinite(var_daily) else np.nan
        predicted_vol_rows.append(pred_vol_row)
        if cfg.verbose:
            print(f"{d.date()} -> {execution_date.date()} signals loaded from pkl")

    target_theme_weights = pd.DataFrame(target_theme_rows)
    target_stock_weights = pd.DataFrame(target_stock_rows)
    if len(target_theme_weights):
        target_theme_weights.index = pd.MultiIndex.from_tuples(target_theme_weights.index, names=["execution_date", "method"])
    if len(target_stock_weights):
        target_stock_weights.index = pd.MultiIndex.from_tuples(target_stock_weights.index, names=["execution_date", "method"])

    diagnostics_by_rebalance = {
        "coherence_z": pd.DataFrame(coherence_z_rows).sort_index() if coherence_z_rows else pd.DataFrame(),
        "coherence_p": pd.DataFrame(coherence_p_rows).sort_index() if coherence_p_rows else pd.DataFrame(),
        "coherence_q": pd.DataFrame(coherence_q_rows).sort_index() if coherence_q_rows else pd.DataFrame(),
        "momentum_score": pd.DataFrame(momentum_rows).sort_index() if momentum_rows else pd.DataFrame(),
        "theme_importance": pd.DataFrame(importance_rows).sort_index() if importance_rows else pd.DataFrame(),
        "purification_adjusted_r2": pd.DataFrame(pur_r2_rows).sort_index() if pur_r2_rows else pd.DataFrame(),
        "covariance_diagnostics": pd.DataFrame(cov_diag_rows).sort_index() if cov_diag_rows else pd.DataFrame(),
        "predicted_volatility": pd.DataFrame(predicted_vol_rows).sort_index() if predicted_vol_rows else pd.DataFrame(),
    }

    monthly_returns = (1.0 + net).resample("ME").prod() - 1.0
    metrics = performance_metrics(net, gross, turnover, transaction_cost, cfg.annualization)
    return {
        "input_source": "precomputed_pkl",
        "precompute_metadata": precomputed_bundle["metadata"],
        "rebalance_dates": rebalance_dates,
        "daily_returns_gross": gross,
        "daily_returns_net": net,
        "daily_turnover": turnover,
        "daily_transaction_cost": transaction_cost,
        "daily_cash_weight": cash_weight,
        "target_theme_weights": target_theme_weights,
        "target_stock_weights": target_stock_weights,
        "optimization_log": pd.DataFrame(opt_logs),
        "diagnostics_by_rebalance": diagnostics_by_rebalance,
        "monthly_returns_net": monthly_returns,
        "metrics": metrics,
    }


def assert_precomputed_equivalence(
    recomputed_result: Mapping[str, Any],
    precomputed_result: Mapping[str, Any],
    atol: float = 1e-10,
) -> None:
    for key in ["daily_returns_net", "target_theme_weights", "target_stock_weights"]:
        left = recomputed_result[key].sort_index().sort_index(axis=1)
        right = precomputed_result[key].sort_index().sort_index(axis=1)
        pd.testing.assert_frame_equal(left, right, check_dtype=False, atol=atol, rtol=0.0)


## 任意: 逐次再計算版との等価性検証

小さい期間または代表リバランス日で、pklロード版と逐次再計算版が同じ中間入力・バックテスト結果を返すか確認する。既定では実行しない。


In [ ]:
def _assert_frame_close(left: pd.DataFrame, right: pd.DataFrame, name: str, atol: float) -> None:
    left = left.sort_index().sort_index(axis=1)
    right = right.sort_index().sort_index(axis=1)
    pd.testing.assert_frame_equal(left, right, check_dtype=False, atol=atol, rtol=0.0, obj=name)


def assert_precomputed_rebalance_input_equivalence(
    signal_date: pd.Timestamp,
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    precomputed_bundle: Mapping[str, Any],
    atol: float = 1e-10,
) -> None:
    sequential = compute_rebalance_inputs(signal_date, data, stores, cfg)
    precomputed = compute_rebalance_inputs_from_precomputed(signal_date, data, cfg, precomputed_bundle)
    _assert_frame_close(sequential["A_t"], precomputed["A_t"], "A_t", atol)
    _assert_frame_close(sequential["Z_t"], precomputed["Z_t"], "Z_t", atol)
    _assert_frame_close(sequential["Omega"], precomputed["Omega"], "Omega", atol)
    _assert_frame_close(
        sequential["payoff_result"]["theme_returns_g"],
        precomputed["payoff_result"]["theme_returns_g"],
        "theme_returns_g",
        atol,
    )
    _assert_frame_close(
        sequential["payoff_result"]["theme_residuals_e"],
        precomputed["payoff_result"]["theme_residuals_e"],
        "theme_residuals_e",
        atol,
    )


def run_optional_precomputed_equivalence_check(
    backtest_result: Mapping[str, Any],
    data: Mapping[str, Any],
    stores: Stores,
    cfg: BacktestConfig,
    methods: Sequence[str],
    cash_returns: pd.Series,
    theme_eligibility: Optional[pd.Series],
    precomputed_bundle: Mapping[str, Any],
) -> None:
    if precomputed_bundle is None:
        raise ValueError("precomputed_bundle is required for equivalence check.")
    first_rebalance = pd.Timestamp(backtest_result["rebalance_dates"][0])
    assert_precomputed_rebalance_input_equivalence(first_rebalance, data, stores, cfg, precomputed_bundle, cfg.equivalence_atol)
    recomputed_result = run_backtest_allocation_methods(
        data=data,
        stores=stores,
        cfg=cfg,
        methods=methods,
        cash_returns=cash_returns,
        theme_eligibility=theme_eligibility,
    )
    assert_precomputed_equivalence(recomputed_result, backtest_result, cfg.equivalence_atol)
    print("Precomputed pkl equivalence check passed.")


## パフォーマンス評価・表示


In [ ]:
def performance_metrics(
    net_returns: pd.DataFrame,
    gross_returns: pd.DataFrame,
    turnover: pd.DataFrame,
    transaction_cost: pd.DataFrame,
    annualization: int = 252,
) -> pd.DataFrame:
    rows = {}
    for col in net_returns.columns:
        r = net_returns[col].dropna()
        if r.empty:
            continue
        wealth = (1.0 + r).cumprod()
        years = len(r) / annualization
        ann_return = wealth.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 and wealth.iloc[-1] > 0 else np.nan
        ann_vol = r.std(ddof=1) * math.sqrt(annualization)
        sharpe = r.mean() / r.std(ddof=1) * math.sqrt(annualization) if r.std(ddof=1) > 0 else np.nan
        downside = r[r < 0].std(ddof=1) * math.sqrt(annualization)
        sortino = (r.mean() * annualization) / downside if downside > 0 else np.nan
        dd = wealth / wealth.cummax() - 1.0
        max_dd = dd.min()
        calmar = ann_return / abs(max_dd) if max_dd < 0 else np.nan
        gross = gross_returns[col].reindex(r.index)
        rows[col] = {
            "cumulative_return_net": wealth.iloc[-1] - 1.0,
            "cumulative_return_gross": (1.0 + gross.fillna(0.0)).cumprod().iloc[-1] - 1.0,
            "annualized_return": ann_return,
            "annualized_volatility": ann_vol,
            "sharpe": sharpe,
            "sortino": sortino,
            "max_drawdown": max_dd,
            "calmar": calmar,
            "skew": r.skew(),
            "kurtosis": r.kurt(),
            "daily_win_rate": float((r > 0).mean()),
            "average_daily_turnover": float(turnover[col].reindex(r.index).fillna(0.0).mean()),
            "total_turnover": float(turnover[col].reindex(r.index).fillna(0.0).sum()),
            "total_transaction_cost": float(transaction_cost[col].reindex(r.index).fillna(0.0).sum()),
        }
    return pd.DataFrame(rows).T


def plot_backtest_comparison(backtest_result: Mapping[str, Any]) -> None:
    import matplotlib.pyplot as plt

    net = backtest_result["daily_returns_net"]
    if net.empty:
        print("No net returns to plot.")
        return
    wealth = (1.0 + net).cumprod()
    ax = wealth.plot(figsize=(12, 4), title="Cumulative net return")
    ax.set_ylabel("Growth of 1")
    plt.show()

    drawdown = wealth / wealth.cummax() - 1.0
    ax = drawdown.plot(figsize=(12, 3), title="Net drawdown")
    ax.set_ylabel("Drawdown")
    plt.show()

    ax = backtest_result["daily_turnover"].resample("ME").sum().plot(figsize=(12, 3), title="Monthly one-way turnover")
    ax.set_ylabel("Turnover")
    plt.show()

    ax = backtest_result["daily_transaction_cost"].resample("ME").sum().plot(figsize=(12, 3), title="Monthly transaction cost")
    ax.set_ylabel("Return drag")
    plt.show()

    ax = backtest_result["daily_cash_weight"].plot(figsize=(12, 3), title="Daily cash weight")
    ax.set_ylabel("Cash weight")
    plt.show()


In [ ]:
RUN_EQUIVALENCE_CHECK = False
if RUN_EQUIVALENCE_CHECK:
    run_optional_precomputed_equivalence_check(
        backtest_result=backtest_result,
        data=data,
        stores=stores,
        cfg=cfg,
        methods=allocation_methods,
        cash_returns=cash_returns,
        theme_eligibility=theme_eligibility,
        precomputed_bundle=precomputed_bundle,
    )


## 実行セル


In [ ]:
raw_inputs = load_raw_inputs_from_paths_or_existing(paths, cfg, existing_inputs)
data = validate_and_normalise_inputs(raw_inputs, cfg)
stores = build_stores(data)
cash_returns = load_optional_cash_returns(paths, existing_inputs, data["stock_returns"].index, cfg.cash_return_default_daily)
theme_eligibility = load_optional_theme_eligibility(paths, existing_inputs)

RUN_PRECOMPUTE = False
if RUN_PRECOMPUTE or cfg.force_recompute_precomputed:
    precompute_outputs = precompute_item10_artifacts(data, stores, cfg, cfg.precompute_dir)
    display(pd.Series(precompute_outputs["metadata"]).to_frame("value"))

if cfg.require_precomputed_inputs:
    precomputed_bundle = load_precomputed_bundle(cfg.precompute_dir, cfg, data)
    backtest_result = run_backtest_allocation_methods_precomputed(
        data=data,
        cfg=cfg,
        methods=allocation_methods,
        cash_returns=cash_returns,
        theme_eligibility=theme_eligibility,
        precomputed_bundle=precomputed_bundle,
    )
else:
    precomputed_bundle = None
    backtest_result = run_backtest_allocation_methods(
        data=data,
        stores=stores,
        cfg=cfg,
        methods=allocation_methods,
        cash_returns=cash_returns,
        theme_eligibility=theme_eligibility,
    )
    backtest_result["input_source"] = "sequential_recompute"

print("input_source:", backtest_result["input_source"])
if precomputed_bundle is not None:
    print("precompute_dir:", precomputed_bundle["precompute_dir"])
print("rebalance dates:", len(backtest_result["rebalance_dates"]))
print("daily_returns_net shape:", backtest_result["daily_returns_net"].shape)
print("target_theme_weights shape:", backtest_result["target_theme_weights"].shape)
print("target_stock_weights shape:", backtest_result["target_stock_weights"].shape)


## 出力確認


In [ ]:
display(backtest_result["metrics"])
if "precompute_metadata" in backtest_result:
    display(pd.Series(backtest_result["precompute_metadata"]).to_frame("value"))
display(backtest_result["daily_returns_net"].tail())
display(backtest_result["daily_turnover"].tail())
display(backtest_result["daily_transaction_cost"].tail())
display(backtest_result["daily_cash_weight"].tail())
display(backtest_result["optimization_log"].tail(30))

diag = backtest_result["diagnostics_by_rebalance"]
for name, frame in diag.items():
    print(name, frame.shape)
    display(frame.tail())


## グラフ


In [ ]:
plot_backtest_comparison(backtest_result)


## 受入チェック

- pklロード版として実行されている
- すべての約定日はリバランス日より後である
- グロス・ネットリターン、turnover、取引費用、キャッシュ比率が同じ営業日・同じ手法列で出力される
- `target_stock_weights = A_t @ theta_t` の再構成誤差が十分小さい
- 制約付き手法はテーマ上限、銘柄上限、cash制約を満たす
- Lee-Liu制約なし手法は `sum(theta)≈1` と `theta.T @ a≈S_T` を満たす
- pkl metadata と現在設定の `config_signature` が一致する


In [ ]:

if cfg.require_precomputed_inputs:
    assert backtest_result.get("input_source") == "precomputed_pkl", "Backtest must run from precomputed pkl inputs."
    assert precomputed_bundle["metadata"]["config_signature"] == precompute_config_signature(cfg), "precompute config_signature mismatch."
    validate_precomputed_bundle(precomputed_bundle, data, cfg)

methods = list(allocation_methods)
for key in ["daily_returns_gross", "daily_returns_net", "daily_turnover", "daily_transaction_cost", "daily_cash_weight"]:
    frame = backtest_result[key]
    assert not frame.empty, f"{key} is empty."
    assert list(frame.columns) == methods, f"{key} columns must match allocation_methods."
    assert frame.index.equals(backtest_result["daily_returns_net"].index), f"{key} index mismatch."

opt_log = backtest_result["optimization_log"].copy()
assert not opt_log.empty, "optimization_log is empty."
if {"signal_date", "execution_date"}.issubset(opt_log.columns):
    dated = opt_log.dropna(subset=["signal_date", "execution_date"]).copy()
    dated["signal_date"] = pd.to_datetime(dated["signal_date"])
    dated["execution_date"] = pd.to_datetime(dated["execution_date"])
    assert (dated["execution_date"] > dated["signal_date"]).all(), "execution_date must be after signal_date."

if "stock_weight_reconstruction_error" in opt_log.columns:
    err = pd.to_numeric(opt_log["stock_weight_reconstruction_error"], errors="coerce").dropna()
    assert err.max() < 1e-8, "target_stock_weights must equal A_t @ theta_t."

lee = opt_log[opt_log.get("method", pd.Series(index=opt_log.index, dtype=object)) == "lee_liu_unconstrained"]
if not lee.empty:
    if "budget_error" in lee.columns:
        assert pd.to_numeric(lee["budget_error"], errors="coerce").abs().dropna().max() < 1e-6, "Lee-Liu budget error is too large."
    if "score_error" in lee.columns:
        assert pd.to_numeric(lee["score_error"], errors="coerce").abs().dropna().max() < 1e-6, "Lee-Liu score error is too large."

for method in ["constrained_realworld", "direct_alpha"]:
    subset = opt_log[opt_log.get("method", pd.Series(index=opt_log.index, dtype=object)) == method]
    ok = subset[~subset.get("status", "").astype(str).str.contains("error", na=False)] if not subset.empty else subset
    if ok.empty:
        continue
    if "budget_plus_cash" in ok.columns:
        assert pd.to_numeric(ok["budget_plus_cash"], errors="coerce").dropna().le(1.0 + 1e-6).all(), f"{method} budget constraint violated."
    if "cash_weight" in ok.columns:
        assert pd.to_numeric(ok["cash_weight"], errors="coerce").dropna().ge(-1e-8).all(), f"{method} cash constraint violated."
    if "max_theme_weight" in ok.columns:
        assert pd.to_numeric(ok["max_theme_weight"], errors="coerce").dropna().le(cfg.theme_cap + 1e-6).all(), f"{method} theme cap violated."
    if "max_stock_weight" in ok.columns:
        assert pd.to_numeric(ok["max_stock_weight"], errors="coerce").dropna().le(cfg.stock_cap + 1e-6).all(), f"{method} stock cap violated."

print("Item 10 acceptance checks passed.")


## 任意: 結果保存

必要な場合だけ `EXPORT_RESULTS=True` にする。既定ではファイルを書き出さない。


In [ ]:
EXPORT_RESULTS = False
output_dir = Path("data/phase_b/backtest_outputs")

if EXPORT_RESULTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    backtest_result["daily_returns_gross"].to_csv(output_dir / "daily_returns_gross.csv")
    backtest_result["daily_returns_net"].to_csv(output_dir / "daily_returns_net.csv")
    backtest_result["daily_turnover"].to_csv(output_dir / "daily_turnover.csv")
    backtest_result["daily_transaction_cost"].to_csv(output_dir / "daily_transaction_cost.csv")
    backtest_result["daily_cash_weight"].to_csv(output_dir / "daily_cash_weight.csv")
    backtest_result["target_theme_weights"].to_csv(output_dir / "target_theme_weights.csv")
    backtest_result["target_stock_weights"].to_csv(output_dir / "target_stock_weights.csv")
    backtest_result["optimization_log"].to_csv(output_dir / "optimization_log.csv", index=False)
    backtest_result["metrics"].to_csv(output_dir / "metrics.csv")
    backtest_result["monthly_returns_net"].to_csv(output_dir / "monthly_returns_net.csv")
    pd.to_pickle(backtest_result["precompute_metadata"], output_dir / "precompute_metadata.pkl")
    print(f"exported to {output_dir}")
